In [1]:
from src.data_loader import prepare_futures_data
import pandas as pd
import matplotlib.pyplot as plt
import datetime as dt
import numpy as np
from src.data_loader import prepare_futures_data
from src.bar_builder import build_15min_bars
from src.signal import add_high_volume_breakout_signal

In [2]:
df = prepare_futures_data("data/RB9999.XSGE.csv")

## First simple hypothesis
The direction of the most recent minute's return is related to the direction of the return for the following minute

In [3]:
start_date_train = dt.datetime(year=2015, month=1, day=1)
end_date_train = dt.datetime(year=2018, month=12, day=31, hour=23, minute=59, second=59)
research_df = df[(df['date'] >= start_date_train) & (df['date'] <= end_date_train)].copy()
research_df["next_return_1m"] = (
    research_df
    .groupby("session_block")["clean_return_1m"]
    .shift(-1)
)
correlation = research_df["clean_return_1m"].corr(
    research_df["next_return_1m"]
)

print(correlation)

-0.11545596415994033


In [4]:
start_date_validation = dt.datetime(year=2019, month=1, day=1)
end_date_validation = dt.datetime(year=2019, month=12, day=31, hour=23, minute=59, second=59)
validation_df = df[(df['date'] >= start_date_validation) & (df['date'] <= end_date_validation)].copy()
validation_df["next_return_1m"] = (
    validation_df
    .groupby("session_block")["clean_return_1m"]
    .shift(-1)
)

RB one-minute returns exhibited a negative lag-one correlation of −0.116 during 2015–2018, providing preliminary evidence of short-term return reversal

In [5]:
def check_corr(df, year):
    start_date = dt.datetime(year=year, month=1, day=1)
    end_date = dt.datetime(year=year, month=12, day=31, hour=23, minute=59, second=59)
    research_df = df[(df['date'] >= start_date) & (df['date'] <= end_date)].copy()
    research_df["next_return_1m"] = (
        research_df
        .groupby("session_block")["clean_return_1m"]
        .shift(-1)
    )
    correlation = research_df["clean_return_1m"].corr(
        research_df["next_return_1m"]
    )

    return correlation

print(check_corr(df, 2015))
print(check_corr(df, 2016))
print(check_corr(df, 2017))
print(check_corr(df, 2018))

-0.1656758896178106
-0.11961615068680378
-0.10072677590340637
-0.08500855903162408


The negative lag-one correlation persisted in every training year, but its magnitude declined steadily from −0.166 in 2015 to −0.085 in 2018, suggesting that the reversal effect may be weakening over time

In [6]:
print(check_corr(df, 2019))

-0.11125237868528126


The reversal effect weakened between 2015 and 2018, but remained present out of sample in 2019, where the lag-one correlation recovered to −0.111

In [7]:
research_df["return_direction"] = (
    np.sign(research_df["clean_return_1m"])
    .map({
        -1: "negative",
         0: "zero",
         1: "positive",
    })
)

conditional_returns = (
    research_df
    .groupby("return_direction")
    .agg(
        observations=("next_return_1m", "count"),
        mean_next_return=("next_return_1m", "mean"),
    )
)

conditional_returns["mean_next_return_bps"] = (
    conditional_returns["mean_next_return"] * 10_000
)
print(conditional_returns)

                  observations  mean_next_return  mean_next_return_bps
return_direction                                                      
negative                138145      1.257504e-04              1.257504
positive                137283     -1.258904e-04             -1.258904
zero                     87283     -3.164559e-07             -0.003165


In [8]:
validation_df["return_direction"] = (
    np.sign(validation_df["clean_return_1m"])
    .map({
        -1: "negative",
         0: "zero",
         1: "positive",
    })
)

conditional_returns = (
    validation_df
    .groupby("return_direction")
    .agg(
        observations=("next_return_1m", "count"),
        mean_next_return=("next_return_1m", "mean"),
    )
)

conditional_returns["mean_next_return_bps"] = (
    conditional_returns["mean_next_return"] * 10_000
)
print(conditional_returns)

                  observations  mean_next_return  mean_next_return_bps
return_direction                                                      
negative                 32689          0.000081              0.812433
positive                 32695         -0.000081             -0.806423
zero                     16018          0.000011              0.106958


RB one-minute returns exhibit a stable short-term reversal effect. The lag-one correlation was negative in every training year from 2015 to 2018 and remained negative out of sample in 2019. After a positive minute, the next return averaged −1.26 bps in training and −0.81 bps in validation; after a negative minute, it averaged +1.26 bps and +0.81 bps, respectively. However, an RB round trip costs approximately 2 bps in commissions and at least one tick of spread, resulting in total estimated costs of about 4.86 bps at a price of 3,500 CNY. Since the validation gross edge is only about 0.81 bps, trading every one-minute reversal is not viable

In [9]:
magnitude_df = research_df[(research_df['clean_return_1m'].notna()) 
                        & (research_df['clean_return_1m'] != 0) &
                        (research_df['next_return_1m'].notna())].copy()

magnitude_df["abs_return_1m"] = (
    magnitude_df["clean_return_1m"].abs()
)
magnitude_df["return_quintile"] = pd.qcut(
    magnitude_df["abs_return_1m"],
    q=5,
    labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
)
print(magnitude_df["return_quintile"].value_counts().sort_index())
print(magnitude_df.groupby(
    "return_quintile",
    observed=True,
)["abs_return_1m"].agg(["count", "min", "max"]))

return_quintile
Q1    55162
Q2    55133
Q3    54976
Q4    55099
Q5    55058
Name: count, dtype: int64
                 count       min       max
return_quintile                           
Q1               55162  0.000227  0.000399
Q2               55133  0.000399  0.000512
Q3               54976  0.000512  0.000730
Q4               55099  0.000731  0.001099
Q5               55058  0.001099  0.030196


In [10]:
magnitude_df['reversal_payoff'] = -np.sign(magnitude_df['clean_return_1m']) * magnitude_df['next_return_1m']

group_magnitude_df = magnitude_df.groupby('return_quintile', observed=True).agg(
                    count_observation=('return_quintile', 'size'),
                    mean_reversal_payoff=('reversal_payoff', 'mean'),
)
group_magnitude_df['mean_reversal_payoff_bps'] = group_magnitude_df['mean_reversal_payoff'] * 10_000
print(group_magnitude_df)

                 count_observation  mean_reversal_payoff  \
return_quintile                                            
Q1                           55162              0.000072   
Q2                           55133              0.000108   
Q3                           54976              0.000129   
Q4                           55099              0.000147   
Q5                           55058              0.000173   

                 mean_reversal_payoff_bps  
return_quintile                            
Q1                               0.721071  
Q2                               1.083280  
Q3                               1.290024  
Q4                               1.472367  
Q5                               1.725407  


In [11]:
group_magnitude_df['net_payoff_bps'] = group_magnitude_df['mean_reversal_payoff_bps'] - 4.86
print(group_magnitude_df)

                 count_observation  mean_reversal_payoff  \
return_quintile                                            
Q1                           55162              0.000072   
Q2                           55133              0.000108   
Q3                           54976              0.000129   
Q4                           55099              0.000147   
Q5                           55058              0.000173   

                 mean_reversal_payoff_bps  net_payoff_bps  
return_quintile                                            
Q1                               0.721071       -4.138929  
Q2                               1.083280       -3.776720  
Q3                               1.290024       -3.569976  
Q4                               1.472367       -3.387633  
Q5                               1.725407       -3.134593  


The magnitude analysis revealed a monotonic relationship: larger one-minute price moves were followed by stronger reversals. Gross reversal payoff increased from 0.72 bps in the lowest quintile to 1.73 bps in the highest. However, none of the quintiles exceeded the estimated minimum round-trip cost of 4.86 bps. The best group still produced an estimated net payoff of −3.13 bps. Therefore, the one-minute reversal signal is statistically persistent but not economically tradable using market orders. This hypothesis is rejected in its current form

In [12]:
df['group_15min'] = (
    df.groupby('session_block')['date']
      .transform(lambda x: ((x - x.min()).dt.total_seconds() // (15 * 60)).astype(int))
)
print(df.head(35))

                  date    open    high     low   close   volume        money  \
0  2009-03-27 09:01:00  3550.0  3662.0  3550.0  3660.0  10528.0  378617760.0   
1  2009-03-27 09:02:00  3651.0  3661.0  3582.0  3582.0   6326.0  228733500.0   
2  2009-03-27 09:03:00  3588.0  3632.0  3588.0  3609.0   4760.0  172252600.0   
3  2009-03-27 09:04:00  3608.0  3626.0  3596.0  3602.0   5180.0  186904140.0   
4  2009-03-27 09:05:00  3602.0  3604.0  3583.0  3600.0   5220.0  187556440.0   
5  2009-03-27 09:06:00  3600.0  3602.0  3596.0  3601.0   2908.0  104680460.0   
6  2009-03-27 09:07:00  3600.0  3602.0  3563.0  3570.0   4014.0  143684640.0   
7  2009-03-27 09:08:00  3572.0  3600.0  3570.0  3598.0   3834.0  137539080.0   
8  2009-03-27 09:09:00  3599.0  3600.0  3580.0  3588.0   2802.0  100558940.0   
9  2009-03-27 09:10:00  3588.0  3593.0  3585.0  3588.0   1540.0   55273640.0   
10 2009-03-27 09:11:00  3590.0  3602.0  3588.0  3592.0   3792.0  136398500.0   
11 2009-03-27 09:12:00  3592.0  3596.0  

In [13]:
df_15_min = df.groupby(['session_block', 'group_15min']).agg(
    start=('date', 'min'),
    end=('date', 'max'),
    open=('open', 'first'),
    high=('high', 'max'),
    low=('low', 'min'),
    close=('close', 'last'),
    volume=('volume', 'sum'),
    money=('money','sum'),
    open_interest=('open_interest', 'last'),
    symnol=('symbol', 'first'),
    number_of_minutes=('date', 'count')
)
df_15_min = df_15_min[df_15_min['number_of_minutes'] == 15]
df_15_min = df_15_min.reset_index()
print(df_15_min.shape)
print(df_15_min.head())

(61619, 13)
   session_block  group_15min               start                 end    open  \
0              0            0 2009-03-27 09:01:00 2009-03-27 09:15:00  3550.0   
1              0            1 2009-03-27 09:16:00 2009-03-27 09:30:00  3586.0   
2              0            2 2009-03-27 09:31:00 2009-03-27 09:45:00  3603.0   
3              0            3 2009-03-27 09:46:00 2009-03-27 10:00:00  3592.0   
4              0            4 2009-03-27 10:01:00 2009-03-27 10:15:00  3585.0   

     high     low   close   volume         money  open_interest  symnol  \
0  3662.0  3550.0  3587.0  60762.0  2.186263e+09        25674.0  RB0909   
1  3623.0  3582.0  3602.0  40496.0  1.457586e+09        34082.0  RB0909   
2  3606.0  3591.0  3592.0  25956.0  9.344330e+08        41316.0  RB0909   
3  3597.0  3581.0  3586.0  26536.0  9.524709e+08        45822.0  RB0909   
4  3595.0  3583.0  3593.0  14954.0  5.369493e+08        46468.0  RB0909   

   number_of_minutes  
0                 15  
1   

In [14]:
df_15_min['return_15m'] = df_15_min['close'] / df_15_min['open'] - 1
df_15_min["next_return_15m"] = (
    df_15_min
    .groupby("session_block")["return_15m"]
    .shift(-1)
)
print(df_15_min.head())

   session_block  group_15min               start                 end    open  \
0              0            0 2009-03-27 09:01:00 2009-03-27 09:15:00  3550.0   
1              0            1 2009-03-27 09:16:00 2009-03-27 09:30:00  3586.0   
2              0            2 2009-03-27 09:31:00 2009-03-27 09:45:00  3603.0   
3              0            3 2009-03-27 09:46:00 2009-03-27 10:00:00  3592.0   
4              0            4 2009-03-27 10:01:00 2009-03-27 10:15:00  3585.0   

     high     low   close   volume         money  open_interest  symnol  \
0  3662.0  3550.0  3587.0  60762.0  2.186263e+09        25674.0  RB0909   
1  3623.0  3582.0  3602.0  40496.0  1.457586e+09        34082.0  RB0909   
2  3606.0  3591.0  3592.0  25956.0  9.344330e+08        41316.0  RB0909   
3  3597.0  3581.0  3586.0  26536.0  9.524709e+08        45822.0  RB0909   
4  3595.0  3583.0  3593.0  14954.0  5.369493e+08        46468.0  RB0909   

   number_of_minutes  return_15m  next_return_15m  
0         

## Second hypothesis
Is the direction of the current 15-minute movement related to the movement of the next 15-minute bar?

In [15]:
df_15_min_train = df_15_min[(df_15_min['start'] >= start_date_train) & (df_15_min['start'] <= end_date_train)].copy()
correlation = df_15_min_train["return_15m"].corr(
    df_15_min_train["next_return_15m"]
)


print(correlation)

-0.008680362292536353


In [16]:
def check_corr_15m(df, year):
    start_date = dt.datetime(year=year, month=1, day=1)
    end_date = dt.datetime(year=year, month=12, day=31, hour=23, minute=59, second=59)
    research_df = df[(df['start'] >= start_date) & (df['start'] <= end_date)].copy()
    df_15_min["next_return_15m"] = (
    df_15_min
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )
    correlation = research_df["return_15m"].corr(
        research_df["next_return_15m"]
    )
    return correlation
print(check_corr_15m(df_15_min_train, 2015))
print(check_corr_15m(df_15_min_train, 2016))
print(check_corr_15m(df_15_min_train, 2017))
print(check_corr_15m(df_15_min_train, 2018))
print(check_corr_15m(df_15_min, 2019))

-0.0026134662313660453
-0.024085953685196324
-0.01069839785657447
0.03814920959668888
-0.014783004464515375


The lag-one correlation of non-overlapping 15-minute returns was weak and unstable. It ranged from −0.024 to +0.038 across 2015–2018 and changed sign in 2018. The 2019 validation correlation was −0.015. Therefore, the immediately preceding 15-minute return does not provide a robust momentum or reversal signal

## Third hypothesis
Does the cumulative return of the last 60 minutes predict the return of the next 15 minutes?

In [17]:
first_open_60m = (
    df_15_min
    .groupby("session_block")["open"]
    .shift(3)
)

df_15_min["past_return_60m"] = (
    df_15_min["close"] / first_open_60m - 1
)
df_15_min_train = df_15_min[(df_15_min['start'] >= start_date_train) & (df_15_min['start'] <= end_date_train)].copy()
print(df_15_min.head())

   session_block  group_15min               start                 end    open  \
0              0            0 2009-03-27 09:01:00 2009-03-27 09:15:00  3550.0   
1              0            1 2009-03-27 09:16:00 2009-03-27 09:30:00  3586.0   
2              0            2 2009-03-27 09:31:00 2009-03-27 09:45:00  3603.0   
3              0            3 2009-03-27 09:46:00 2009-03-27 10:00:00  3592.0   
4              0            4 2009-03-27 10:01:00 2009-03-27 10:15:00  3585.0   

     high     low   close   volume         money  open_interest  symnol  \
0  3662.0  3550.0  3587.0  60762.0  2.186263e+09        25674.0  RB0909   
1  3623.0  3582.0  3602.0  40496.0  1.457586e+09        34082.0  RB0909   
2  3606.0  3591.0  3592.0  25956.0  9.344330e+08        41316.0  RB0909   
3  3597.0  3581.0  3586.0  26536.0  9.524709e+08        45822.0  RB0909   
4  3595.0  3583.0  3593.0  14954.0  5.369493e+08        46468.0  RB0909   

   number_of_minutes  return_15m  next_return_15m  past_return

In [18]:
print(len(df_15_min_train[(df_15_min_train['past_return_60m'].notna()) & (df_15_min_train['next_return_15m'].notna())]))

9209


In [19]:
correlation = df_15_min_train["past_return_60m"].corr(
    df_15_min_train["next_return_15m"]
)
print(correlation)

0.014202456129143604


In [20]:
def check_corr_15m(df, year):
    start_date = dt.datetime(year=year, month=1, day=1)
    end_date = dt.datetime(year=year, month=12, day=31, hour=23, minute=59, second=59)
    df = df[(df['start'] >= start_date) & (df['start'] <= end_date)].copy()
    observations = len(df[(df['past_return_60m'].notna()) & (df['next_return_15m'].notna())])
    correlation = df["past_return_60m"].corr(
        df["next_return_15m"]
    )

    return correlation, observations

rows = []

for year in range(2015, 2020):
    correlation, observations = check_corr_15m(df_15_min, year)

    rows.append({
        'year': year,
        'observations': observations,
        'correlation': correlation
    })

table = pd.DataFrame(rows)

table


,year,observations,correlation
0,2015,3564,0.014813
1,2016,2288,0.020609
2,2017,1684,0.005040
3,2018,1673,0.011377
4,2019,1680,-0.000932


The 60-minute return showed a weak positive relationship with the following 15-minute return in every training year. However, the effect disappeared out of sample in 2019, where the correlation was effectively zero. Therefore, this momentum hypothesis did not pass validation

## Fourth Hypothesis
If the current 15-minute bar closes above the high of the previous 60 minutes, the next bar tends to continue rising. If it closes below the low of the previous 60 minutes, the next bar tends to continue falling

In [21]:
df_15_min["prior_60m_high"] = (
    df_15_min
    .groupby("session_block")["high"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(window=4, min_periods=4)
            .max()
        )
    )
)

df_15_min["prior_60m_low"] = (
    df_15_min
    .groupby("session_block")["low"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(window=4, min_periods=4)
            .min()
        )
    )
)
print(df_15_min.head())

   session_block  group_15min               start                 end    open  \
0              0            0 2009-03-27 09:01:00 2009-03-27 09:15:00  3550.0   
1              0            1 2009-03-27 09:16:00 2009-03-27 09:30:00  3586.0   
2              0            2 2009-03-27 09:31:00 2009-03-27 09:45:00  3603.0   
3              0            3 2009-03-27 09:46:00 2009-03-27 10:00:00  3592.0   
4              0            4 2009-03-27 10:01:00 2009-03-27 10:15:00  3585.0   

     high     low   close   volume         money  open_interest  symnol  \
0  3662.0  3550.0  3587.0  60762.0  2.186263e+09        25674.0  RB0909   
1  3623.0  3582.0  3602.0  40496.0  1.457586e+09        34082.0  RB0909   
2  3606.0  3591.0  3592.0  25956.0  9.344330e+08        41316.0  RB0909   
3  3597.0  3581.0  3586.0  26536.0  9.524709e+08        45822.0  RB0909   
4  3595.0  3583.0  3593.0  14954.0  5.369493e+08        46468.0  RB0909   

   number_of_minutes  return_15m  next_return_15m  past_return

In [22]:
df_15_min['breakout_signal'] = np.select(
    [
        df_15_min['close'] > df_15_min['prior_60m_high'],
        df_15_min['close'] < df_15_min['prior_60m_low'],
    ],
    [
        1,
        -1
    ],
    default=0
)
df_15_min_train = df_15_min.loc[
    (df_15_min["start"] >= start_date_train)
    & (df_15_min["start"] <= end_date_train)
    & df_15_min["prior_60m_high"].notna()
    & df_15_min["prior_60m_low"].notna()
    & df_15_min["next_return_15m"].notna()
].copy()
print(df_15_min_train['breakout_signal'].value_counts())

breakout_signal
 0    5270
 1     603
-1     439
Name: count, dtype: int64


In [23]:
df_15_min_train['breakout_payoff'] = df_15_min_train['breakout_signal'] * df_15_min_train['next_return_15m']
breakout_trades = df_15_min_train[df_15_min_train['breakout_signal'] != 0].copy()
df_15_min_statistic = breakout_trades.groupby('breakout_signal').agg(
    count=('breakout_signal', 'size'),
    mean_bps=('breakout_payoff', 'mean'),
)
df_15_min_statistic['mean_bps'] *= 10_000
win_rate = len(breakout_trades[breakout_trades['breakout_payoff'] > 0]) / len(breakout_trades['breakout_payoff'])
print(win_rate)
print(df_15_min_statistic)

0.4692898272552783
                 count  mean_bps
breakout_signal                 
-1                 439  4.739988
 1                 603  3.763393


In [24]:
payoff_stats = pd.Series({
    'mean_payoff_bps': breakout_trades['breakout_payoff'].mean() * 10000,

    'mean_positive_payoff_bps': breakout_trades.loc[
        breakout_trades['breakout_payoff'] > 0,
        'breakout_payoff'
    ].mean() * 10000,

    'mean_negative_payoff_bps': breakout_trades.loc[
        breakout_trades['breakout_payoff'] < 0,
        'breakout_payoff'
    ].mean() * 10000,

    'zero_payoff_count': (
        breakout_trades['breakout_payoff'] == 0
    ).sum()
})

payoff_stats

mean_payoff_bps              4.174837
mean_positive_payoff_bps    30.350437
mean_negative_payoff_bps   -22.856608
zero_payoff_count           94.000000
dtype: float64

## Continuation of the hypothesis
A breakout on high volume is more reliable than a breakout on low volume

In [25]:
df_15_min["prior_60m_mean_volume"] = (
    df_15_min
    .groupby("session_block")["volume"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(window=4, min_periods=4)
            .mean()
        )
    )
)
df_15_min['volume_ratio'] = df_15_min['volume'] / df_15_min['prior_60m_mean_volume']
print(df_15_min.head())

   session_block  group_15min               start                 end    open  \
0              0            0 2009-03-27 09:01:00 2009-03-27 09:15:00  3550.0   
1              0            1 2009-03-27 09:16:00 2009-03-27 09:30:00  3586.0   
2              0            2 2009-03-27 09:31:00 2009-03-27 09:45:00  3603.0   
3              0            3 2009-03-27 09:46:00 2009-03-27 10:00:00  3592.0   
4              0            4 2009-03-27 10:01:00 2009-03-27 10:15:00  3585.0   

     high     low   close   volume         money  ...  symnol  \
0  3662.0  3550.0  3587.0  60762.0  2.186263e+09  ...  RB0909   
1  3623.0  3582.0  3602.0  40496.0  1.457586e+09  ...  RB0909   
2  3606.0  3591.0  3592.0  25956.0  9.344330e+08  ...  RB0909   
3  3597.0  3581.0  3586.0  26536.0  9.524709e+08  ...  RB0909   
4  3595.0  3583.0  3593.0  14954.0  5.369493e+08  ...  RB0909   

  number_of_minutes  return_15m  next_return_15m  past_return_60m  \
0                15    0.010423         0.004462     

In [26]:
breakout_trades = df_15_min.loc[
    (df_15_min["start"] >= start_date_train)
    & (df_15_min["start"] <= end_date_train)
    & df_15_min["volume_ratio"].notna() &
    (df_15_min['breakout_signal'] != 0)
    & df_15_min["next_return_15m"].notna()
].copy()

breakout_trades['breakout_payoff'] = breakout_trades['breakout_signal'] * breakout_trades['next_return_15m']
breakout_trades['volume_ratio_high_1'] = breakout_trades['volume_ratio'] > 1
statistic = breakout_trades.groupby('volume_ratio_high_1').agg(
    count=('breakout_payoff', 'size'),
    mean_payoff_bps=('breakout_payoff', 'mean'),
    win_rate=(
        "breakout_payoff",
        lambda x: (x > 0).mean(),
    ),
)
statistic['mean_payoff_bps'] *= 10_000
print(statistic)


                     count  mean_payoff_bps  win_rate
volume_ratio_high_1                                  
False                  246        -0.429213  0.382114
True                   796         5.597697  0.496231


In [27]:
high_volume_breakouts = df_15_min.loc[
    (df_15_min["start"] >= "2015-01-01")
    & (df_15_min["start"] < "2020-01-01")
    & (df_15_min["breakout_signal"] != 0)
    & (df_15_min["volume_ratio"] > 1)
    & df_15_min["next_return_15m"].notna()
].copy()

high_volume_breakouts["year"] = (
    high_volume_breakouts["start"].dt.year
)

high_volume_breakouts["breakout_payoff"] = (
    high_volume_breakouts["breakout_signal"]
    * high_volume_breakouts["next_return_15m"]
)


yearly_results = (
    high_volume_breakouts
    .groupby("year")
    .agg(
        trades=("breakout_payoff", "size"),
        gross_payoff=("breakout_payoff", "mean"),
        win_rate=(
            "breakout_payoff",
            lambda x: (x > 0).mean(),
        ),
    )
)
yearly_results["gross_payoff_bps"] = (
    yearly_results["gross_payoff"] * 10_000
)

yearly_results["net_payoff_bps"] = (
    yearly_results["gross_payoff_bps"] - 4.86
)

yearly_results["win_rate_pct"] = (
    yearly_results["win_rate"] * 100
)

yearly_results[
[
        "trades",
        "gross_payoff_bps",
        "net_payoff_bps",
        "win_rate_pct",
    ]
]

,trades,gross_payoff_bps,net_payoff_bps,win_rate_pct
year,,,,
2015,267,3.261272,-1.598728,42.322097
2016,207,7.380889,2.520889,57.971014
2017,184,8.460419,3.600419,51.086957
2018,138,3.626423,-1.233577,49.275362
2019,142,-0.064241,-4.924241,41.549296


In [28]:
universe = {
    "RB": "data/RB9999.XSGE.csv",
    "CU": "data/CU9999.XSGE.csv",
    "AU": "data/AU9999.XSGE.csv",
    "RU": "data/RU9999.XSGE.csv",
    "M": "data/M9999.XDCE.csv",
    "Y": "data/Y9999.XDCE.csv",
    "IF": "data/IF9999.CCFX.csv",
    "TA": "data/TA9999.XZCE.csv",
}


bars_by_instrument = {}
summary_rows = []

for instrument, path in universe.items():
    minute_df = prepare_futures_data(path)
    bars_df = build_15min_bars(minute_df)

    bars_df["instrument"] = instrument
    bars_by_instrument[instrument] = bars_df

    summary_rows.append({
        "instrument": instrument,
        "number_of_15m_bars": len(bars_df),
        "first_date": bars_df["start"].min(),
        "last_date": bars_df["end"].max(),
    })

    del minute_df


universe_summary = pd.DataFrame(summary_rows)
print(universe_summary)

  instrument  number_of_15m_bars          first_date           last_date
0         RB               61619 2009-03-27 09:01:00 2021-10-18 15:00:00
1         CU               89792 2005-01-04 09:01:00 2021-10-18 15:00:00
2         AU               91961 2008-01-10 09:01:00 2021-10-18 15:00:00
3         RU               74576 2005-01-04 09:01:00 2021-10-18 15:00:00
4          M               76538 2005-01-04 09:01:00 2021-10-18 15:00:00
5          Y               72863 2006-01-09 09:01:00 2021-10-18 15:00:00
6         IF               47496 2010-04-16 09:16:00 2021-10-18 15:00:00
7         TA               68858 2006-12-18 09:01:00 2021-10-18 15:00:00


In [29]:
summary_rows = []
signals_by_instrument = {}

for instrument, bars_df in bars_by_instrument.items():
    signal_df = add_high_volume_breakout_signal(
        bars_df,
        lookback=4,
    )

    signal_df["return_15m"] = (
        signal_df["close"] / signal_df["open"] - 1
    )

    signal_df["next_return_15m"] = (
        signal_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    signals_by_instrument[instrument] = signal_df

    train_signals = signal_df.loc[
        (signal_df["start"] >= start_date_train)
        & (signal_df["start"] <= end_date_train)
        & (signal_df["strategy_signal"] != 0)
        & signal_df["next_return_15m"].notna()
    ]

    train_signals = train_signals.copy()
    train_signals["strategy_payoff"] = (
        train_signals["strategy_signal"]
        * train_signals["next_return_15m"]
    )

    summary_rows.append({
        "instrument": instrument,
        "total_signals": len(train_signals),
        "long_signals": (train_signals["strategy_signal"] == 1).sum(),
        "short_signals": (train_signals["strategy_signal"] == -1).sum(),
        "gross_payoff_bps": train_signals["strategy_payoff"].mean() * 10_000,
        "win_rate_pct": (train_signals["strategy_payoff"] > 0).mean() * 100,
    })

signal_summary = pd.DataFrame(summary_rows)
print(signal_summary)

  instrument  total_signals  long_signals  short_signals  gross_payoff_bps  \
0         RB            796           463            333          5.597697   
1         CU           1443           734            709          0.775233   
2         AU           2021           990           1031          0.225616   
3         RU            547           260            287          4.240441   
4          M            864           424            440          0.168614   
5          Y            838           379            459          0.847958   
6         IF           1262           603            659          3.226942   
7         TA            757           368            389          2.016137   

   win_rate_pct  
0     49.623116  
1     41.857242  
2     39.435923  
3     41.133455  
4     41.319444  
5     42.720764  
6     49.049128  
7     42.932629  


In [30]:
validation_rows = []

for instrument, signal_df in signals_by_instrument.items():
    validation_trades = signal_df.loc[
        (signal_df["start"] >= start_date_validation)
        & (signal_df["start"] <= end_date_validation)
        & (signal_df["strategy_signal"] != 0)
        & signal_df["next_return_15m"].notna()
    ].copy()

    validation_trades["strategy_payoff"] = (
        validation_trades["strategy_signal"]
        * validation_trades["next_return_15m"]
    )

    validation_rows.append({
        "instrument": instrument,
        "total_signals": len(validation_trades),
        "long_signals": (validation_trades["strategy_signal"] == 1).sum(),
        "short_signals": (validation_trades["strategy_signal"] == -1).sum(),
        "gross_payoff_bps": (
            validation_trades["strategy_payoff"].mean() * 10_000
        ),
        "win_rate_pct": (
            (validation_trades["strategy_payoff"] > 0).mean() * 100
        ),
    })

validation_summary = pd.DataFrame(validation_rows)
print(validation_summary)

  instrument  total_signals  long_signals  short_signals  gross_payoff_bps  \
0         RB            142            70             72         -0.064241   
1         CU            374           194            180         -0.810977   
2         AU            506           265            241          0.698965   
3         RU            114            32             82          1.148982   
4          M            130            64             66         -1.572936   
5          Y            130            53             77         -1.345806   
6         IF            300           153            147          0.616918   
7         TA            197           104             93         -0.209603   

   win_rate_pct  
0     41.549296  
1     39.572193  
2     40.909091  
3     38.596491  
4     32.307692  
5     36.923077  
6     46.333333  
7     38.071066  


In [31]:
comparison = signal_summary[
    ["instrument", "total_signals", "gross_payoff_bps", "win_rate_pct"]
].merge(
    validation_summary[
        ["instrument", "total_signals", "gross_payoff_bps", "win_rate_pct"]
    ],
    on="instrument",
    suffixes=("_train", "_validation"),
)

comparison["payoff_change_bps"] = (
    comparison["gross_payoff_bps_validation"]
    - comparison["gross_payoff_bps_train"]
)

print(comparison)

  instrument  total_signals_train  gross_payoff_bps_train  win_rate_pct_train  \
0         RB                  796                5.597697           49.623116   
1         CU                 1443                0.775233           41.857242   
2         AU                 2021                0.225616           39.435923   
3         RU                  547                4.240441           41.133455   
4          M                  864                0.168614           41.319444   
5          Y                  838                0.847958           42.720764   
6         IF                 1262                3.226942           49.049128   
7         TA                  757                2.016137           42.932629   

   total_signals_validation  gross_payoff_bps_validation  \
0                       142                    -0.064241   
1                       374                    -0.810977   
2                       506                     0.698965   
3                       114   

The high-volume breakout signal produced positive gross returns across all eight instruments during 2015–2018. However, its performance deteriorated for seven of eight instruments in the 2019 validation period. The remaining positive validation payoffs were small and unlikely to cover transaction costs. Therefore, the apparent training-period effect was neither temporally stable nor economically tradable, and the strategy is rejected in its current form

In [32]:
momentum_rows = []
momentum_by_instrument = {}

for instrument, bars_df in bars_by_instrument.items():
    momentum_df = bars_df.copy()

    momentum_df["open_60m_ago"] = (
        momentum_df
        .groupby("session_block")["open"]
        .shift(3)
    )

    momentum_df["past_return_60m"] = (
        momentum_df["close"] / momentum_df["open_60m_ago"] - 1
    )

    momentum_df["return_15m"] = (
        momentum_df["close"] / momentum_df["open"] - 1
    )

    momentum_df["next_return_15m"] = (
        momentum_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    momentum_by_instrument[instrument] = momentum_df

    train_df = momentum_df.loc[
        (momentum_df["start"] >= start_date_train)
        & (momentum_df["start"] <= end_date_train)
        & momentum_df["past_return_60m"].notna()
        & momentum_df["next_return_15m"].notna()
    ]

    momentum_rows.append({
        "instrument": instrument,
        "observations": len(train_df),
        "correlation": train_df["past_return_60m"].corr(
            train_df["next_return_15m"]
        ),
    })

momentum_summary = pd.DataFrame(momentum_rows)
print(momentum_summary)

  instrument  observations  correlation
0         RB          9209     0.014202
1         CU         14289     0.022309
2         AU         19971     0.011151
3         RU          7713     0.045786
4          M          9549     0.008295
5          Y          9549     0.003532
6         IF          8276     0.012695
7         TA          8607    -0.020400


In [33]:
qcut_rows = []

for instrument, momentum_df in momentum_by_instrument.items():
    qcut_df = momentum_df.copy()
    train_df = qcut_df.loc[
        (qcut_df["start"] >= start_date_train)
        & (qcut_df["start"] <= end_date_train)
        & qcut_df["past_return_60m"].notna()
        & qcut_df["next_return_15m"].notna()
    ].copy()

    train_df["return_quintile"] = pd.qcut(
        abs(train_df["past_return_60m"]),
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    )

    train_df["momentum_payoff"] = (
        np.sign(train_df["past_return_60m"])
        * train_df["next_return_15m"]
    )
    statistic_qcut = train_df.groupby(
        "return_quintile",
        observed=True).agg(
            count_obs=('start', 'count'),
            mean_next_return_15m=('next_return_15m', 'mean'),
            mean_momentum_payoff=("momentum_payoff", "mean"),
    )
    statistic_qcut["mean_momentum_payoff_bps"] = (
        statistic_qcut["mean_momentum_payoff"] * 10_000
    )
    statistic_qcut["instrument"] = instrument
    qcut_rows.append(statistic_qcut.reset_index())


qcut_summary = pd.concat(qcut_rows, ignore_index=True)
print(qcut_summary)


   return_quintile  count_obs  mean_next_return_15m  mean_momentum_payoff  \
0               Q1       1842              0.000006              0.000121   
1               Q2       1842             -0.000051              0.000061   
2               Q3       1841              0.000154              0.000154   
3               Q4       1842              0.000135              0.000096   
4               Q5       1842              0.000089              0.000121   
5               Q1       2858              0.000042             -0.000013   
6               Q2       2858             -0.000022              0.000053   
7               Q3       2857             -0.000019              0.000011   
8               Q4       2858              0.000008             -0.000046   
9               Q5       2858              0.000017              0.000090   
10              Q1       3995              0.000019              0.000007   
11              Q2       3994             -0.000017              0.000028   

In [34]:
quintile_cross_market = (
    qcut_summary
    .groupby("return_quintile", observed=True)
    .agg(
        instrument_count=("instrument", "nunique"),
        mean_payoff_bps=("mean_momentum_payoff_bps", "mean"),
        median_payoff_bps=("mean_momentum_payoff_bps", "median"),
        min_payoff_bps=("mean_momentum_payoff_bps", "min"),
        max_payoff_bps=("mean_momentum_payoff_bps", "max"),
    )
)

print(quintile_cross_market)

                 instrument_count  mean_payoff_bps  median_payoff_bps  \
return_quintile                                                         
Q1                              8         0.364712           0.340715   
Q2                              8         0.366898           0.406119   
Q3                              8         0.479717           0.332031   
Q4                              8         0.079858           0.116309   
Q5                              8         0.994869           0.637019   

                 min_payoff_bps  max_payoff_bps  
return_quintile                                  
Q1                    -0.135475        1.212895  
Q2                    -0.264845        0.995385  
Q3                    -1.203717        2.006866  
Q4                    -0.831960        0.959378  
Q5                    -0.287728        4.411274  


## Fifth hypothesis
Trade in the direction of the previous 60-minute movement only if its absolute magnitude exceeds the instrument's 80th percentile calculated on the training set

In [35]:
threshold_rows = []

for instrument, momentum_df in momentum_by_instrument.items():
    train_df = momentum_df.loc[
        (momentum_df["start"] >= start_date_train)
        & (momentum_df["start"] <= end_date_train)
        & momentum_df["past_return_60m"].notna()
    ]

    threshold_80 = (
        train_df["past_return_60m"]
        .abs()
        .quantile(0.80)
    )

    threshold_rows.append({
        "instrument": instrument,
        "threshold_80": threshold_80,
        "threshold_80_pct": threshold_80 * 100,
    })

momentum_thresholds = pd.DataFrame(threshold_rows)

print(momentum_thresholds)

  instrument  threshold_80  threshold_80_pct
0         RB      0.006214          0.621427
1         CU      0.003653          0.365260
2         AU      0.001751          0.175101
3         RU      0.006980          0.698002
4          M      0.004027          0.402693
5          Y      0.003588          0.358809
6         IF      0.007316          0.731578
7         TA      0.004664          0.466360


In [36]:
validation_rows = []

for instrument, momentum_df in momentum_by_instrument.items():
    validation_df = momentum_df.loc[
        (momentum_df["start"] >= start_date_validation)
        & (momentum_df["start"] <= end_date_validation)
        & momentum_df["past_return_60m"].notna()
        & momentum_df["next_return_15m"].notna()
    ].copy()

    threshold_80 = momentum_thresholds.loc[momentum_thresholds['instrument'] == instrument, 'threshold_80'].iloc[0]
    validation_df['signal'] = np.where(
        validation_df['past_return_60m'].abs() > threshold_80,
        np.sign(validation_df['past_return_60m']),
        0
    )
    validation_df = validation_df[validation_df['signal'] != 0].copy()
    validation_df['momentum_payoff'] = validation_df['signal'] * validation_df['next_return_15m']
    validation_rows.append({
        "instrument": instrument,
        "threshold_80_pct": threshold_80 * 100,
        "trades": len(validation_df),
        "long_trades": (validation_df["signal"] == 1).sum(),
        "short_trades": (validation_df["signal"] == -1).sum(),
        "gross_payoff_bps": (
            validation_df["momentum_payoff"].mean() * 10_000
        ),
        "win_rate_pct": (
            (validation_df["momentum_payoff"] > 0).mean() * 100
        ),
    })

momentum_validation_summary = pd.DataFrame(validation_rows)
print(momentum_validation_summary)

  instrument  threshold_80_pct  trades  long_trades  short_trades  \
0         RB          0.621427     224          123           101   
1         CU          0.365260     233          124           109   
2         AU          0.175101    1195          594           601   
3         RU          0.698002     179           84            95   
4          M          0.402693     339          152           187   
5          Y          0.358809     300          146           154   
6         IF          0.731578     239          140            99   
7         TA          0.466360     552          279           273   

   gross_payoff_bps  win_rate_pct  
0         -0.435017     43.303571  
1         -0.512863     41.630901  
2          0.104852     43.179916  
3         -3.368639     38.547486  
4          0.086563     41.887906  
5         -0.728053     40.333333  
6          4.741487     53.974895  
7          1.796744     44.021739  


In [37]:
if_bars = bars_by_instrument["IF"]

price_columns = ["open", "high", "low", "close"]

for column in price_columns:
    scaled_price = if_bars[column] / 0.2

    valid_tick = np.isclose(
        scaled_price,
        np.round(scaled_price),
    )

    print(
        column,
        "violations:",
        (~valid_tick).sum(),
    )


if_minute = prepare_futures_data("data/IF9999.CCFX.csv")

close_changes = (
    if_minute
    .groupby("session_block")["close"]
    .diff()
    .abs()
)

positive_changes = close_changes[
    close_changes > 1e-10
]

print(positive_changes.min())

open violations: 0
high violations: 0
low violations: 0
close violations: 0
0.1999999999998181


In [38]:
if_minute_2019 = if_minute[(if_minute['date'] >= start_date_validation) & (if_minute['date'] <= end_date_validation)]
median_if_close = if_minute_2019['close'].median()
tick_cost = 0.2 / median_if_close * 10000
print(tick_cost)

0.5241914347119568


In [39]:
validation_rows = []
instrument = 'IF'
momentum_df = momentum_by_instrument['IF']
validation_df = momentum_df.loc[
    (momentum_df["start"] >= start_date_validation)
    & (momentum_df["start"] <= end_date_validation)
    & momentum_df["past_return_60m"].notna()
    & momentum_df["next_return_15m"].notna()
].copy()

threshold_80 = momentum_thresholds.loc[momentum_thresholds['instrument'] == instrument, 'threshold_80'].iloc[0]
validation_df['signal'] = np.where(
    validation_df['past_return_60m'].abs() > threshold_80,
    np.sign(validation_df['past_return_60m']),
    0
)
validation_df = validation_df[validation_df['signal'] != 0].copy()
validation_df['momentum_payoff'] = validation_df['signal'] * validation_df['next_return_15m']
val_standart_error = validation_df['momentum_payoff'].std() / np.sqrt(len(validation_df))
validation_rows.append({
    "instrument": instrument,
    "threshold_80_pct": threshold_80 * 100,
    "trades": len(validation_df),
    "mean_payoff": validation_df['momentum_payoff'].mean(),
    "std_payoff": validation_df['momentum_payoff'].std(),
    "standart_error": val_standart_error,
    "t-statistic": validation_df['momentum_payoff'].mean() / val_standart_error
})
validation_rows = pd.DataFrame(validation_rows)
print(validation_rows)

  instrument  threshold_80_pct  trades  mean_payoff  std_payoff  \
0         IF          0.731578     239     0.000474    0.003394   

   standart_error  t-statistic  
0         0.00022      2.15954  


A 60-minute momentum effect was investigated across eight futures markets. The strongest 20% of historical movements did not produce a consistent cross-market effect. IF was the only instrument with a relatively stable result: its gross payoff was 4.41 bps during 2015–2018 and 4.74 bps during the 2019 validation period, with a validation t-statistic of 2.16. However, the estimated intraday commission and one-tick spread reduced the expected net payoff to approximately 0.54 bps. Therefore, the signal is statistically interesting before costs but is not economically robust enough to trade in its current form.  The hypothesis is rejected as a standalone strategy

## A New Hypothesis
Price movements accompanied by rising open interest are more likely to continue, whereas movements accompanied by falling open interest are more likely to reverse

In [40]:
oi_by_instrument = {}

for instrument, bars_df in bars_by_instrument.items():
    oi_df = bars_df.copy()

    oi_df["return_15m"] = (
        oi_df["close"] / oi_df["open"] - 1
    )

    oi_df["open_interest_pct_change"] = (
        oi_df
        .groupby("session_block")["open_interest"]
        .pct_change(fill_method=None)
    )

    oi_df["next_return_15m"] = (
        oi_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    oi_by_instrument[instrument] = oi_df


rb_oi = oi_by_instrument["RB"]

print(rb_oi.head())
print(rb_oi.tail())

   session_block  group_15min               start                 end    open  \
0              0            0 2009-03-27 09:01:00 2009-03-27 09:15:00  3550.0   
1              0            1 2009-03-27 09:16:00 2009-03-27 09:30:00  3586.0   
2              0            2 2009-03-27 09:31:00 2009-03-27 09:45:00  3603.0   
3              0            3 2009-03-27 09:46:00 2009-03-27 10:00:00  3592.0   
4              0            4 2009-03-27 10:01:00 2009-03-27 10:15:00  3585.0   

     high     low   close   volume         money  open_interest  symbol  \
0  3662.0  3550.0  3587.0  60762.0  2.186263e+09        25674.0  RB0909   
1  3623.0  3582.0  3602.0  40496.0  1.457586e+09        34082.0  RB0909   
2  3606.0  3591.0  3592.0  25956.0  9.344330e+08        41316.0  RB0909   
3  3597.0  3581.0  3586.0  26536.0  9.524709e+08        45822.0  RB0909   
4  3595.0  3583.0  3593.0  14954.0  5.369493e+08        46468.0  RB0909   

   number_of_minutes instrument  return_15m  open_interest_pct

In [41]:
oi_rows = []
for instrument, oi_df in oi_by_instrument.items():
    oi_train_df = oi_df[(oi_df['start'] >= start_date_train) & (oi_df['start'] <= end_date_train)].copy()
    count_zero_oi = len(oi_train_df[oi_train_df['open_interest_pct_change'] == 0])
    count_NaN_oi = len(oi_train_df[oi_train_df['open_interest_pct_change'].isna()])
    median_oi_change = oi_train_df['open_interest_pct_change'].median()
    min_oi_change = oi_train_df['open_interest_pct_change'].min()
    max_oi_change = oi_train_df['open_interest_pct_change'].max()
    threshold_99 = (
        oi_train_df["open_interest_pct_change"]
        .quantile(0.99)
    )
    threshold_1 = (
        oi_train_df["open_interest_pct_change"]
        .quantile(0.01)
    )
    oi_rows.append({
        "instrument": instrument,
        "threshold_99": threshold_99,
        "threshold_1": threshold_1,
        "count_zero_oi": count_zero_oi,
        "count_NaN_oi": count_NaN_oi,
        "median_oi_change": median_oi_change,
        "min_oi_change": min_oi_change,
        "max_oi_change": max_oi_change
    })


oi_rows = pd.DataFrame(oi_rows)
print(oi_rows)

  instrument  threshold_99  threshold_1  count_zero_oi  count_NaN_oi  \
0         RB      0.017141    -0.023078              2          3872   
1         CU      0.013506    -0.016133             41          3872   
2         AU      0.009093    -0.009552             72          3872   
3         RU      0.018615    -0.028022            923          3872   
4          M      0.012328    -0.013156             16          3871   
5          Y      0.010862    -0.011288             27          3871   
6         IF      0.039653    -0.087198             32          1962   
7         TA      0.021982    -0.021178             13          3872   

   median_oi_change  min_oi_change  max_oi_change  
0         -0.000191      -0.085580       0.052320  
1         -0.000596      -0.050166       0.049783  
2         -0.000152      -0.032425       0.054726  
3         -0.000245      -0.076337       0.072853  
4         -0.000286      -0.066080       0.046141  
5         -0.000225      -0.039894     

In [42]:
regime_rows = []
oi_regimes_by_instrument = {}

for instrument, oi_df in oi_by_instrument.items():
    train_df = oi_df.loc[
        (oi_df["start"] >= start_date_train)
        & (oi_df["start"] <= end_date_train)
        & oi_df["return_15m"].notna()
        & oi_df["open_interest_pct_change"].notna()
        & oi_df["next_return_15m"].notna()
        & (oi_df["return_15m"] != 0)
        & (oi_df["open_interest_pct_change"] != 0)
    ].copy()

    train_df["regime"] = np.select(
        [
            (train_df["return_15m"] > 0)
            & (train_df["open_interest_pct_change"] > 0),

            (train_df["return_15m"] < 0)
            & (train_df["open_interest_pct_change"] > 0),

            (train_df["return_15m"] > 0)
            & (train_df["open_interest_pct_change"] < 0),

            (train_df["return_15m"] < 0)
            & (train_df["open_interest_pct_change"] < 0),
        ],
        [
            "price_up_oi_up",
            "price_down_oi_up",
            "price_up_oi_down",
            "price_down_oi_down",
        ],
        default="unclassified",
    )

    oi_regimes_by_instrument[instrument] = train_df

    regime_statistics = (
        train_df
        .groupby("regime")
        .agg(
            observations=("next_return_15m", "count"),
            mean_next_return=("next_return_15m", "mean"),
        )
        .reset_index()
    )

    regime_statistics["mean_next_return_bps"] = (
        regime_statistics["mean_next_return"] * 10_000
    )

    regime_statistics["instrument"] = instrument
    regime_rows.append(regime_statistics)

oi_regime_summary = pd.concat(
    regime_rows,
    ignore_index=True,
)

oi_regime_summary = oi_regime_summary[
    [
        "instrument",
        "regime",
        "observations",
        "mean_next_return",
        "mean_next_return_bps",
    ]
]

print(oi_regime_summary)

   instrument              regime  observations  mean_next_return  \
0          RB  price_down_oi_down          3600          0.000070   
1          RB    price_down_oi_up          3944          0.000042   
2          RB    price_up_oi_down          3527         -0.000029   
3          RB      price_up_oi_up          4123          0.000060   
4          CU  price_down_oi_down          5573         -0.000016   
5          CU    price_down_oi_up          4382          0.000006   
6          CU    price_up_oi_down          5633          0.000010   
7          CU      price_up_oi_up          4426         -0.000006   
8          AU  price_down_oi_down          6164          0.000006   
9          AU    price_down_oi_up          5509         -0.000014   
10         AU    price_up_oi_down          5739         -0.000007   
11         AU      price_up_oi_up          6026          0.000012   
12         RU  price_down_oi_down          3420         -0.000003   
13         RU    price_down_oi_up 

In [43]:
price_up_comparison = (
    oi_regime_summary
    .loc[
        oi_regime_summary["regime"].isin([
            "price_up_oi_down",
            "price_up_oi_up",
        ])
    ]
    .pivot(
        index="instrument",
        columns="regime",
        values="mean_next_return_bps",
    )
    .reset_index()
)

price_up_comparison["oi_down_minus_oi_up_bps"] = (
    price_up_comparison["price_up_oi_down"]
    - price_up_comparison["price_up_oi_up"]
)

print(price_up_comparison)
share_negative = (
    price_up_comparison["oi_down_minus_oi_up_bps"] < 0
).mean()

print("Share with negative OI effect:", share_negative)

regime instrument  price_up_oi_down  price_up_oi_up  oi_down_minus_oi_up_bps
0              AU         -0.068997        0.120676                -0.189672
1              CU          0.097010       -0.061281                 0.158291
2              IF         -1.548809       -0.821894                -0.726915
3               M         -0.537697       -0.542803                 0.005106
4              RB         -0.285614        0.598455                -0.884069
5              RU         -1.741993       -0.888603                -0.853390
6              TA         -1.389717       -0.784695                -0.605022
7               Y         -0.844497       -0.519682                -0.324816
Share with negative OI effect: 0.75


The previous hypothesis was not confirmed. New hypothesis:
If the current 15-minute bar is positive and open interest has decreased compared to the previous bar, open a short position for the next 15-minute bar. Otherwise, do not trade.

In [44]:
oi_validation_rows = []

for instrument, oi_df in oi_by_instrument.items():
    validation_df = oi_df.loc[
        (oi_df["start"] >= start_date_validation)
        & (oi_df["start"] <= end_date_validation)
        & oi_df["return_15m"].notna()
        & oi_df["open_interest_pct_change"].notna()
        & oi_df["next_return_15m"].notna()
        & (oi_df["return_15m"] > 0)
        & (oi_df["open_interest_pct_change"] != 0)
    ].copy()

    price_up_oi_down = validation_df.loc[
        validation_df["open_interest_pct_change"] < 0
    ].copy()

    price_up_oi_up = validation_df.loc[
        validation_df["open_interest_pct_change"] > 0
    ].copy()

    # После price_up + OI_down наша позиция — short.
    price_up_oi_down["strategy_payoff"] = (
        -price_up_oi_down["next_return_15m"]
    )

    mean_next_oi_down_bps = (
        price_up_oi_down["next_return_15m"].mean() * 10_000
    )

    mean_next_oi_up_bps = (
        price_up_oi_up["next_return_15m"].mean() * 10_000
    )

    oi_validation_rows.append({
        "instrument": instrument,
        "trades": len(price_up_oi_down),
        "comparison_oi_up_obs": len(price_up_oi_up),
        "gross_payoff_bps": (
            price_up_oi_down["strategy_payoff"].mean() * 10_000
        ),
        "win_rate_pct": (
            (price_up_oi_down["strategy_payoff"] > 0).mean() * 100
        ),
        "next_return_oi_down_bps": mean_next_oi_down_bps,
        "next_return_oi_up_bps": mean_next_oi_up_bps,
        "oi_down_minus_oi_up_bps": (
            mean_next_oi_down_bps - mean_next_oi_up_bps
        ),
    })

oi_validation_summary = pd.DataFrame(oi_validation_rows)

print(oi_validation_summary)

  instrument  trades  comparison_oi_up_obs  gross_payoff_bps  win_rate_pct  \
0         RB     786                   928          0.756185     50.000000   
1         CU    1266                  1112         -0.285193     43.206951   
2         AU    1461                  1640         -0.105004     44.558522   
3         RU     723                   779          2.103869     49.930844   
4          M     775                   812         -0.317942     43.354839   
5          Y     740                   849          0.726928     48.378378   
6         IF     673                   780         -1.541402     49.628529   
7         TA     879                   968          0.354983     47.440273   

   next_return_oi_down_bps  next_return_oi_up_bps  oi_down_minus_oi_up_bps  
0                -0.756185               1.452895                -2.209080  
1                 0.285193              -0.113369                 0.398562  
2                 0.105004               0.386739                -

During 2015–2018, price increases accompanied by declining open interest were followed by weaker returns than price increases accompanied by rising open interest in six of eight markets. However, this relationship persisted in only three markets during the 2019 validation period. In several instruments, including IF and TA, the relationship reversed. The open-interest condition therefore did not provide stable incremental predictive information, and the hypothesis was rejected as a standalone signal

## New hypothesis
An unusually strong movement in a single 15-minute bar often partially reverses on the following bar

In [45]:
shock_by_instrument = {}

for instrument, bars_df in bars_by_instrument.items():
    shock_df = bars_df.copy()

    shock_df["return_15m"] = (
        shock_df["close"] / shock_df["open"] - 1
    )

    shock_df["next_return_15m"] = (
        shock_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    shock_df["prior_volatility_100"] = (
        shock_df
        .groupby("symbol")["return_15m"]
        .transform(
            lambda x: (
                x.shift(1)
                .rolling(window=100, min_periods=100)
                .std()
            )
        )
    )
    shock_df["shock"] = (
        shock_df["return_15m"]
        / shock_df["prior_volatility_100"]
    )

    shock_by_instrument[instrument] = shock_df

rb_shock = shock_by_instrument["RB"]

print(rb_shock.head())
print(rb_shock.tail())

   session_block  group_15min               start                 end    open  \
0              0            0 2009-03-27 09:01:00 2009-03-27 09:15:00  3550.0   
1              0            1 2009-03-27 09:16:00 2009-03-27 09:30:00  3586.0   
2              0            2 2009-03-27 09:31:00 2009-03-27 09:45:00  3603.0   
3              0            3 2009-03-27 09:46:00 2009-03-27 10:00:00  3592.0   
4              0            4 2009-03-27 10:01:00 2009-03-27 10:15:00  3585.0   

     high     low   close   volume         money  open_interest  symbol  \
0  3662.0  3550.0  3587.0  60762.0  2.186263e+09        25674.0  RB0909   
1  3623.0  3582.0  3602.0  40496.0  1.457586e+09        34082.0  RB0909   
2  3606.0  3591.0  3592.0  25956.0  9.344330e+08        41316.0  RB0909   
3  3597.0  3581.0  3586.0  26536.0  9.524709e+08        45822.0  RB0909   
4  3595.0  3583.0  3593.0  14954.0  5.369493e+08        46468.0  RB0909   

   number_of_minutes instrument  return_15m  next_return_15m  

In [46]:
shock_train_rows = []
shock_train_trades_by_instrument = {}

for instrument, shock_df in shock_by_instrument.items():
    train_df = shock_df.loc[
        (shock_df["start"] >= start_date_train)
        & (shock_df["start"] <= end_date_train)
        & shock_df["shock"].notna()
        & shock_df["next_return_15m"].notna()
    ].copy()

    train_df["reversal_signal"] = np.where(
        train_df["shock"].abs() > 2,
        -np.sign(train_df["shock"]),
        0,
    )

    trades = train_df.loc[
        train_df["reversal_signal"] != 0
    ].copy()

    trades["reversal_payoff"] = (
        trades["reversal_signal"]
        * trades["next_return_15m"]
    )

    shock_train_trades_by_instrument[instrument] = trades

    shock_train_rows.append({
        "instrument": instrument,
        "trades": len(trades),
        "long_trades": (trades["reversal_signal"] == 1).sum(),
        "short_trades": (trades["reversal_signal"] == -1).sum(),
        "gross_payoff_bps": (
            trades["reversal_payoff"].mean() * 10_000
        ),
        "win_rate_pct": (
            (trades["reversal_payoff"] > 0).mean() * 100
        ),
    })

shock_train_summary = pd.DataFrame(shock_train_rows)

print(shock_train_summary)

  instrument  trades  long_trades  short_trades  gross_payoff_bps  \
0         RB    1105          581           524         -0.857684   
1         CU    1322          648           674          0.378871   
2         AU    1770          915           855         -0.356680   
3         RU    1058          539           519         -0.741341   
4          M    1244          637           607          0.271093   
5          Y    1166          564           602          1.557906   
6         IF     504          232           272         -1.615524   
7         TA    1120          527           593          0.618712   

   win_rate_pct  
0     48.778281  
1     55.446293  
2     50.847458  
3     53.402647  
4     51.286174  
5     54.802744  
6     50.396825  
7     53.571429  


Extreme 15-minute price shocks did not exhibit a consistent reversal effect across markets. Although several instruments produced win rates above 50%, gross mean payoffs were small, with a maximum of 1.56 bps, while four of eight instruments had negative expected payoffs. The effect was therefore insufficient to cover realistic transaction costs and was rejected without accessing the validation period

## Hypothesis
Following a large 15-minute move (in absolute terms), the subsequent bar also tends to exhibit higher absolute returns.

In [47]:
volatility_rows = []

for instrument, bars_df in bars_by_instrument.items():
    volatility_df = bars_df.copy()

    volatility_df["return_15m"] = (
        volatility_df["close"] / volatility_df["open"] - 1
    )

    volatility_df["next_return_15m"] = (
        volatility_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    train_df = volatility_df.loc[
        (volatility_df["start"] >= start_date_train)
        & (volatility_df["start"] <= end_date_train)
        & volatility_df["return_15m"].notna()
        & volatility_df["next_return_15m"].notna()
    ].copy()

    train_df["abs_return_15m"] = (
        train_df["return_15m"].abs()
    )

    train_df["next_abs_return_15m"] = (
        train_df["next_return_15m"].abs()
    )

    correlation = train_df["abs_return_15m"].corr(
        train_df["next_abs_return_15m"]
    )

    spearman_correlation = train_df["abs_return_15m"].corr(
        train_df["next_abs_return_15m"],
       method="spearman",
    )

    volatility_rows.append({
        "instrument": instrument,
        "observations": len(train_df),
        "abs_return_pearson_correlation": correlation,
        "spearman_correlation": spearman_correlation,
    })

volatility_clustering_summary = pd.DataFrame(volatility_rows)

print(volatility_clustering_summary)

  instrument  observations  abs_return_pearson_correlation  \
0         RB         20825                        0.275809   
1         CU         25905                        0.253091   
2         AU         31587                        0.252005   
3         RU         19329                        0.227808   
4          M         21162                        0.237698   
5          Y         21162                        0.222544   
6         IF         14126                        0.373857   
7         TA         20223                        0.210386   

   spearman_correlation  
0              0.253944  
1              0.165178  
2              0.214952  
3              0.250919  
4              0.164438  
5              0.141348  
6              0.270033  
7              0.157410  


Large 15-minute price movements were followed by increased absolute returns during the subsequent 15-minute interval across all eight instruments. Both Pearson and Spearman correlations were positive, indicating that the result was not driven solely by extreme observations

In [48]:
volatility_validation_rows = []

for instrument, bars_df in bars_by_instrument.items():
    volatility_df = bars_df.copy()

    volatility_df["return_15m"] = (
        volatility_df["close"] / volatility_df["open"] - 1
    )

    volatility_df["next_return_15m"] = (
        volatility_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    validation_df = volatility_df.loc[
        (volatility_df["start"] >= start_date_validation)
        & (volatility_df["start"] <= end_date_validation)
        & volatility_df["return_15m"].notna()
        & volatility_df["next_return_15m"].notna()
    ].copy()

    validation_df["abs_return_15m"] = (
        validation_df["return_15m"].abs()
    )

    validation_df["next_abs_return_15m"] = (
        validation_df["next_return_15m"].abs()
    )

    pearson_correlation = validation_df["abs_return_15m"].corr(
        validation_df["next_abs_return_15m"],
        method="pearson",
    )

    spearman_correlation = validation_df["abs_return_15m"].corr(
        validation_df["next_abs_return_15m"],
        method="spearman",
    )

    volatility_validation_rows.append({
        "instrument": instrument,
        "observations": len(validation_df),
        "pearson_correlation": pearson_correlation,
        "spearman_correlation": spearman_correlation,
    })

volatility_validation_summary = pd.DataFrame(
    volatility_validation_rows
)

print(volatility_validation_summary)

  instrument  observations  pearson_correlation  spearman_correlation
0         RB          4587             0.119918              0.089667
1         CU          6483             0.208014              0.154833
2         AU          7905             0.297681              0.208998
3         RU          4587             0.114813              0.085644
4          M          4699             0.145599              0.119917
5          Y          4699             0.158068              0.073429
6         IF          3416             0.187751              0.150109
7         TA          5033             0.187381              0.133123


Volatility clustering was observed across all eight instruments during the 2015–2018 training period. Both Pearson and Spearman correlations between the current and subsequent absolute 15-minute returns were positive for every market. The relationship remained positive across all instruments in the 2019 validation period, although its magnitude generally weakened. This provides robust evidence that elevated short-term volatility tends to persist across futures markets

In [49]:
volatility_regime_rows = []
volatility_threshold_rows = []

for instrument, bars_df in bars_by_instrument.items():
    volatility_df = bars_df.copy()

    volatility_df["return_15m"] = (
        volatility_df["close"] / volatility_df["open"] - 1
    )

    volatility_df["next_return_15m"] = (
        volatility_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    train_df = volatility_df.loc[
        (volatility_df["start"] >= start_date_train)
        & (volatility_df["start"] <= end_date_train)
        & volatility_df["return_15m"].notna()
        & volatility_df["next_return_15m"].notna()
    ].copy()

    train_df["abs_return_15m"] = (
        train_df["return_15m"].abs()
    )

    train_df["next_abs_return_15m"] = (
        train_df["next_return_15m"].abs()
    )

    threshold_80 = train_df["abs_return_15m"].quantile(0.80)

    train_df["volatility_regime"] = np.where(
        train_df["abs_return_15m"] > threshold_80,
        "high",
        "normal",
    )

    normal_df = train_df.loc[
        train_df["volatility_regime"] == "normal"
    ]

    high_df = train_df.loc[
        train_df["volatility_regime"] == "high"
    ]

    normal_next_abs_return = (
        normal_df["next_abs_return_15m"].mean()
    )

    high_next_abs_return = (
        high_df["next_abs_return_15m"].mean()
    )

    volatility_regime_rows.append({
        "instrument": instrument,
        "threshold_80_pct": threshold_80 * 100,
        "normal_observations": len(normal_df),
        "high_observations": len(high_df),
        "normal_next_abs_return_bps": (
            normal_next_abs_return * 10_000
        ),
        "high_next_abs_return_bps": (
            high_next_abs_return * 10_000
        ),
        "high_to_normal_ratio": (
            high_next_abs_return / normal_next_abs_return
        ),
    })

volatility_regime_summary = pd.DataFrame(
    volatility_regime_rows
)

print(volatility_regime_summary)

  instrument  threshold_80_pct  normal_observations  high_observations  \
0         RB          0.328776                16660               4165   
1         CU          0.183872                20725               5180   
2         AU          0.087166                25269               6318   
3         RU          0.371503                15463               3866   
4          M          0.206597                16929               4233   
5          Y          0.186192                16929               4233   
6         IF          0.360478                11301               2825   
7         TA          0.250389                16178               4045   

   normal_next_abs_return_bps  high_next_abs_return_bps  high_to_normal_ratio  
0                   17.440054                 30.435583              1.745154  
1                   10.100929                 15.719989              1.556291  
2                    4.781800                  8.108947              1.695794  
3            

After an extreme price movement, you cannot use the same position size and stop distance as in normal conditions, because the expected amplitude of the next bar is significantly higher

In [50]:
volatility_regime_validation_rows = []

for instrument, bars_df in bars_by_instrument.items():
    volatility_df = bars_df.copy()

    volatility_df["return_15m"] = (
        volatility_df["close"] / volatility_df["open"] - 1
    )

    volatility_df["next_return_15m"] = (
        volatility_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    validation_df = volatility_df.loc[
        (volatility_df["start"] >= start_date_validation)
        & (volatility_df["start"] <= end_date_validation)
        & volatility_df["return_15m"].notna()
        & volatility_df["next_return_15m"].notna()
    ].copy()

    validation_df["abs_return_15m"] = (
        validation_df["return_15m"].abs()
    )

    validation_df["next_abs_return_15m"] = (
        validation_df["next_return_15m"].abs()
    )

    threshold_80_pct = volatility_regime_summary.loc[
        volatility_regime_summary["instrument"] == instrument,
        "threshold_80_pct",
    ].iloc[0]

    threshold_80 = threshold_80_pct / 100

    validation_df["volatility_regime"] = np.where(
        validation_df["abs_return_15m"] > threshold_80,
        "high",
        "normal",
    )

    normal_df = validation_df.loc[
        validation_df["volatility_regime"] == "normal"
    ]

    high_df = validation_df.loc[
        validation_df["volatility_regime"] == "high"
    ]

    normal_next_abs_return = (
        normal_df["next_abs_return_15m"].mean()
    )

    high_next_abs_return = (
        high_df["next_abs_return_15m"].mean()
    )

    volatility_regime_validation_rows.append({
        "instrument": instrument,
        "threshold_80_pct": threshold_80_pct,
        "normal_observations": len(normal_df),
        "high_observations": len(high_df),
        "high_share_pct": (
            len(high_df) / len(validation_df) * 100
        ),
        "normal_next_abs_return_bps": (
            normal_next_abs_return * 10_000
        ),
        "high_next_abs_return_bps": (
            high_next_abs_return * 10_000
        ),
        "high_to_normal_ratio": (
            high_next_abs_return / normal_next_abs_return
        ),
    })

volatility_regime_validation_summary = pd.DataFrame(
    volatility_regime_validation_rows
)

print(volatility_regime_validation_summary)

  instrument  threshold_80_pct  normal_observations  high_observations  \
0         RB          0.328776                 4013                574   
1         CU          0.183872                 6069                414   
2         AU          0.087166                 6266               1639   
3         RU          0.371503                 4133                454   
4          M          0.206597                 3872                827   
5          Y          0.186192                 3841                858   
6         IF          0.360478                 2979                437   
7         TA          0.250389                 3817               1216   

   high_share_pct  normal_next_abs_return_bps  high_next_abs_return_bps  \
0       12.513625                   14.613134                 19.108731   
1        6.385932                    6.150261                 10.514030   
2       20.733713                    5.017383                  8.817277   
3        9.897537                

The high-volatility regime was successfully validated out of sample. Using the fixed 80th-percentile thresholds estimated from 2015–2018, the mean absolute return of the subsequent 15-minute bar was 1.31–1.76 times larger in the high-volatility regime across all eight instruments during 2019. Although the frequency of high-volatility observations changed across markets, the relationship remained directionally consistent. This feature is suitable for risk forecasting and position sizing, but does not predict return direction by itself

## Hypothesis
Across several markets, the instruments showing the strongest growth over the past 60 minutes may continue to outperform those with the weakest movement over the next 15 minutes

In [51]:
cross_sectional_frames = []

for instrument, bars_df in bars_by_instrument.items():
    instrument_bars = bars_df.copy()
    instrument_bars["instrument"] = instrument

    cross_sectional_frames.append(
        instrument_bars[
            [
                "instrument",
                "session_block",
                "start",
                "end",
                "open",
                "high",
                "low",
                "close",
                "volume",
                "open_interest",
                "symbol",
            ]
        ]
    )

cross_sectional_df = pd.concat(
    cross_sectional_frames,
    ignore_index=True,
)

cross_sectional_df = cross_sectional_df.sort_values(
    ["end", "instrument"]
).reset_index(drop=True)

print(cross_sectional_df.shape)
print(cross_sectional_df.head())

(583703, 11)
  instrument  session_block               start                 end     open  \
0         CU              0 2005-01-04 09:01:00 2005-01-04 09:15:00  29110.0   
1          M              0 2005-01-04 09:01:00 2005-01-04 09:15:00   2150.0   
2         RU              0 2005-01-04 09:01:00 2005-01-04 09:15:00  12040.0   
3         CU              0 2005-01-04 09:16:00 2005-01-04 09:30:00  29050.0   
4          M              0 2005-01-04 09:16:00 2005-01-04 09:30:00   2151.0   

      high      low    close   volume  open_interest  symbol  
0  29170.0  28930.0  29050.0  21644.0       121388.0  CU0503  
1   2154.0   2144.0   2151.0  16870.0        69708.0   M0505  
2  12080.0  12040.0  12045.0    250.0         9292.0  RU0503  
3  29150.0  29050.0  29100.0   6732.0       121902.0  CU0503  
4   2154.0   2150.0   2152.0   3608.0        70230.0   M0505  


In [52]:
duplicate_count = cross_sectional_df.duplicated(
    subset=["instrument", "end"]
).sum()

print("Duplicate instrument/end pairs:", duplicate_count)

Duplicate instrument/end pairs: 0


In [53]:
cross_sectional_train = cross_sectional_df.loc[
    (cross_sectional_df["end"] >= start_date_train)
    & (cross_sectional_df["end"] <= end_date_train)
].copy()

instruments_per_timestamp = (
    cross_sectional_train
    .groupby("end")["instrument"]
    .nunique()
)

print("Distribution of available instruments:")
print(
    instruments_per_timestamp
    .value_counts()
    .sort_index()
)

print("\nSummary:")
print(instruments_per_timestamp.describe())

Distribution of available instruments:
instrument
1     8377
2     3810
3     1598
4      276
5     1276
6      848
7     9524
8    12919
Name: count, dtype: int64

Summary:
count    38628.000000
mean         5.265170
std          2.911741
min          1.000000
25%          2.000000
50%          7.000000
75%          8.000000
max          8.000000
Name: instrument, dtype: float64


In [54]:
cross_sectional_feature_frames = []

for instrument, bars_df in bars_by_instrument.items():
    feature_df = bars_df.copy()
    feature_df["instrument"] = instrument

    feature_df["return_15m"] = (
        feature_df["close"] / feature_df["open"] - 1
    )

    grouped = feature_df.groupby("session_block")

    feature_df["open_60m_ago"] = (
        grouped["open"].shift(3)
    )

    feature_df["group_60m_ago"] = (
        grouped["group_15min"].shift(3)
    )

    valid_past_60m = (
        feature_df["group_15min"]
        - feature_df["group_60m_ago"]
    ).eq(3)

    feature_df["past_return_60m"] = (
        feature_df["close"] / feature_df["open_60m_ago"] - 1
    ).where(valid_past_60m)

    feature_df["next_return_15m"] = (
        grouped["return_15m"].shift(-1)
    )

    feature_df["next_group_15min"] = (
        grouped["group_15min"].shift(-1)
    )

    valid_next_bar = (
        feature_df["next_group_15min"]
        - feature_df["group_15min"]
    ).eq(1)

    feature_df["next_return_15m"] = (
        feature_df["next_return_15m"]
        .where(valid_next_bar)
    )

    cross_sectional_feature_frames.append(
        feature_df[
            [
                "instrument",
                "session_block",
                "group_15min",
                "start",
                "end",
                "symbol",
                "past_return_60m",
                "next_return_15m",
            ]
        ]
    )

cross_sectional_features = pd.concat(
    cross_sectional_feature_frames,
    ignore_index=True,
)

cross_sectional_features = (
    cross_sectional_features
    .sort_values(["end", "instrument"])
    .reset_index(drop=True)
)

In [55]:
cross_sectional_train = cross_sectional_features.loc[
    (cross_sectional_features["end"] >= start_date_train)
    & (cross_sectional_features["end"] <= end_date_train)
    & cross_sectional_features["past_return_60m"].notna()
    & cross_sectional_features["next_return_15m"].notna()
].copy()

available_count = (
    cross_sectional_train
    .groupby("end")["instrument"]
    .transform("nunique")
)

cross_sectional_train_8 = cross_sectional_train.loc[
    available_count == 8
].copy()

print(
    "Common timestamps:",
    cross_sectional_train_8["end"].nunique(),
)

print(
    "Rows:",
    len(cross_sectional_train_8),
)

print(
    cross_sectional_train_8
    .groupby("end")["instrument"]
    .nunique()
    .value_counts()
)

Common timestamps: 1950
Rows: 15600
instrument
8    1950
Name: count, dtype: int64


In [56]:
cross_sectional_rows = []

for timestamp, group in cross_sectional_train_8.groupby("end"):
    max_past_return = group["past_return_60m"].max()
    min_past_return = group["past_return_60m"].min()

    winner_mask = np.isclose(
        group["past_return_60m"],
        max_past_return,
        rtol=1e-10,
        atol=1e-12,
    )

    loser_mask = np.isclose(
        group["past_return_60m"],
        min_past_return,
        rtol=1e-10,
        atol=1e-12,
    )

    winners = group.loc[winner_mask]
    losers = group.loc[loser_mask]

    winner_next_return = winners["next_return_15m"].mean()
    loser_next_return = losers["next_return_15m"].mean()

    cross_sectional_rows.append({
        "end": timestamp,
        "winner_instruments": ",".join(
            sorted(winners["instrument"].tolist())
        ),
        "loser_instruments": ",".join(
            sorted(losers["instrument"].tolist())
        ),
        "winner_count": len(winners),
        "loser_count": len(losers),
        "winner_past_return_60m": max_past_return,
        "loser_past_return_60m": min_past_return,
        "winner_next_return_15m": winner_next_return,
        "loser_next_return_15m": loser_next_return,
        "cross_sectional_payoff": (
            winner_next_return - loser_next_return
        ),
        "all_returns_equal": np.isclose(
            max_past_return,
            min_past_return,
            rtol=1e-10,
            atol=1e-12,
        ),
    })

cross_sectional_results = pd.DataFrame(
    cross_sectional_rows
)

print("Timestamps:", len(cross_sectional_results))

print(
    "Multiple winners:",
    (cross_sectional_results["winner_count"] > 1).sum(),
)

print(
    "Multiple losers:",
    (cross_sectional_results["loser_count"] > 1).sum(),
)

print(
    "All past returns equal:",
    cross_sectional_results["all_returns_equal"].sum(),
)

print(cross_sectional_results.head())

Timestamps: 1950
Multiple winners: 10
Multiple losers: 3
All past returns equal: 0
                  end winner_instruments loser_instruments  winner_count  \
0 2015-01-05 14:30:00                 IF                CU             1   
1 2015-01-05 14:45:00                 IF                 M             1   
2 2015-01-06 14:30:00                 RU                AU             1   
3 2015-01-06 14:45:00                 RU                 Y             1   
4 2015-01-07 14:30:00                 RU                 M             1   

   loser_count  winner_past_return_60m  loser_past_return_60m  \
0            1                0.015414              -0.002652   
1            1                0.022604              -0.002485   
2            1                0.009555               0.000410   
3            1                0.005869              -0.000702   
4            1                0.004884              -0.003141   

   winner_next_return_15m  loser_next_return_15m  cross_sectional_pay

In [57]:
payoffs = (
    cross_sectional_results
    .loc[
        ~cross_sectional_results["all_returns_equal"],
        "cross_sectional_payoff",
    ]
    .dropna()
)

observations = len(payoffs)
mean_payoff = payoffs.mean()
median_payoff = payoffs.median()
std_payoff = payoffs.std()
standard_error = std_payoff / np.sqrt(observations)
t_statistic = mean_payoff / standard_error

cross_sectional_statistics = pd.Series({
    "observations": observations,
    "mean_payoff_bps": mean_payoff * 10_000,
    "median_payoff_bps": median_payoff * 10_000,
    "std_payoff_bps": std_payoff * 10_000,
    "standard_error_bps": standard_error * 10_000,
    "t_statistic": t_statistic,
    "win_rate_pct": (payoffs > 0).mean() * 100,
    "loss_rate_pct": (payoffs < 0).mean() * 100,
    "zero_payoff_count": (payoffs == 0).sum(),
})

print(cross_sectional_statistics)

observations          1950.000000
mean_payoff_bps          3.865525
median_payoff_bps        1.707850
std_payoff_bps          49.730226
standard_error_bps       1.126168
t_statistic              3.432459
win_rate_pct            51.589744
loss_rate_pct           48.051282
zero_payoff_count        7.000000
dtype: float64


In [58]:
yearly_cross_sectional = cross_sectional_results.loc[
    ~cross_sectional_results["all_returns_equal"]
].copy()

yearly_cross_sectional["year"] = (
    yearly_cross_sectional["end"].dt.year
)

yearly_summary = (
    yearly_cross_sectional
    .groupby("year")
    .agg(
        observations=("cross_sectional_payoff", "count"),
        mean_payoff=("cross_sectional_payoff", "mean"),
        median_payoff=("cross_sectional_payoff", "median"),
        std_payoff=("cross_sectional_payoff", "std"),
        win_rate_pct=(
            "cross_sectional_payoff",
            lambda x: (x > 0).mean() * 100,
        ),
    )
)

yearly_summary["standard_error"] = (
    yearly_summary["std_payoff"]
    / np.sqrt(yearly_summary["observations"])
)

yearly_summary["t_statistic"] = (
    yearly_summary["mean_payoff"]
    / yearly_summary["standard_error"]
)

yearly_summary["mean_payoff_bps"] = (
    yearly_summary["mean_payoff"] * 10_000
)

yearly_summary["median_payoff_bps"] = (
    yearly_summary["median_payoff"] * 10_000
)

print(
    yearly_summary[
        [
            "observations",
            "mean_payoff_bps",
            "median_payoff_bps",
            "win_rate_pct",
            "t_statistic",
        ]
    ]
)

      observations  mean_payoff_bps  median_payoff_bps  win_rate_pct  \
year                                                                   
2015           488         5.295864           1.731154     51.229508   
2016           488         4.939550           3.393576     54.098361   
2017           488         3.232006           1.323045     51.434426   
2018           486         1.986980          -0.057235     49.588477   

      t_statistic  
year               
2015     1.674858  
2016     2.473514  
2017     1.662777  
2018     1.250789  


Cross-sectional 60-minute momentum generated a positive gross long-short payoff in every training year. However, the mean payoff declined steadily from 5.30 bps in 2015 to 1.99 bps in 2018, while the median payoff became slightly negative in 2018. Because each observation requires trading both a long and a short leg, the expected transaction costs exceed the gross edge. The strategy was therefore rejected without accessing the validation period.

In [59]:
hc_minute = prepare_futures_data("data/HC9999.XSGE.csv")
hc_15min = build_15min_bars(hc_minute)

print("Shape:", hc_15min.shape)
print("First date:", hc_15min["start"].min())
print("Last date:", hc_15min["end"].max())

duplicate_count = hc_15min.duplicated(
    subset=["session_block", "group_15min"]
).sum()

print("Duplicate blocks:", duplicate_count)

Shape: (43091, 13)
First date: 2014-03-21 09:01:00
Last date: 2021-10-18 15:00:00
Duplicate blocks: 0


In [60]:
rb_15min = bars_by_instrument["RB"].copy()

rb_hc = rb_15min.merge(
    hc_15min,
    on="end",
    how="inner",
    suffixes=("_rb", "_hc"),
)

rb_hc = rb_hc.sort_values("end").reset_index(drop=True)

print("Shape:", rb_hc.shape)
print("First common date:", rb_hc["end"].min())
print("Last common date:", rb_hc["end"].max())

print(
    "Duplicate timestamps:",
    rb_hc["end"].duplicated().sum(),
)

print(
    rb_hc[
        [
            "end",
            "close_rb",
            "close_hc",
            "symbol_rb",
            "symbol_hc",
        ]
    ].head()
)

Shape: (43091, 26)
First common date: 2014-03-21 09:15:00
Last common date: 2021-10-18 15:00:00
Duplicate timestamps: 0
                  end  close_rb  close_hc symbol_rb symbol_hc
0 2014-03-21 09:15:00    3234.0    3302.0    RB1410    HC1410
1 2014-03-21 09:30:00    3235.0    3304.0    RB1410    HC1410
2 2014-03-21 09:45:00    3229.0    3300.0    RB1410    HC1410
3 2014-03-21 10:00:00    3217.0    3298.0    RB1410    HC1410
4 2014-03-21 10:15:00    3228.0    3306.0    RB1410    HC1410


In [61]:
rb_hc = rb_hc.copy()

rb_hc["return_15m_rb"] = (
    rb_hc["close_rb"] / rb_hc["open_rb"] - 1
)

rb_hc["return_15m_hc"] = (
    rb_hc["close_hc"] / rb_hc["open_hc"] - 1
)

session_keys = [
    "session_block_rb",
    "session_block_hc",
]

rb_hc["next_return_15m_rb"] = (
    rb_hc
    .groupby(session_keys)["return_15m_rb"]
    .shift(-1)
)

rb_hc["next_return_15m_hc"] = (
    rb_hc
    .groupby(session_keys)["return_15m_hc"]
    .shift(-1)
)

rb_hc["next_end"] = (
    rb_hc
    .groupby(session_keys)["end"]
    .shift(-1)
)

valid_next_bar = (
    rb_hc["next_end"] - rb_hc["end"]
).eq(pd.Timedelta(minutes=15))

rb_hc["next_return_15m_rb"] = (
    rb_hc["next_return_15m_rb"].where(valid_next_bar)
)

rb_hc["next_return_15m_hc"] = (
    rb_hc["next_return_15m_hc"].where(valid_next_bar)
)

lead_lag_train = rb_hc.loc[
    (rb_hc["end"] >= start_date_train)
    & (rb_hc["end"] <= end_date_train)
].copy()

rb_leads_hc_corr = lead_lag_train["return_15m_rb"].corr(
    lead_lag_train["next_return_15m_hc"]
)

hc_leads_rb_corr = lead_lag_train["return_15m_hc"].corr(
    lead_lag_train["next_return_15m_rb"]
)

print("RB(t) -> HC(t+1):", rb_leads_hc_corr)
print("HC(t) -> RB(t+1):", hc_leads_rb_corr)

print(
    "Valid RB -> HC observations:",
    lead_lag_train[
        ["return_15m_rb", "next_return_15m_hc"]
    ].dropna().shape[0],
)

print(
    "Valid HC -> RB observations:",
    lead_lag_train[
        ["return_15m_hc", "next_return_15m_rb"]
    ].dropna().shape[0],
)

RB(t) -> HC(t+1): -0.01616096106013848
HC(t) -> RB(t+1): 0.015017162091273217
Valid RB -> HC observations: 20825
Valid HC -> RB observations: 20825


In [62]:
def analyze_lead_lag_quintiles(
    df,
    leader_return_col,
    follower_next_return_col,
):
    analysis_df = df.loc[
        df[leader_return_col].notna()
        & df[follower_next_return_col].notna()
        & df[leader_return_col].ne(0)
    ].copy()

    analysis_df["leader_abs_return"] = (
        analysis_df[leader_return_col].abs()
    )

    analysis_df["leader_quintile"] = pd.qcut(
        analysis_df["leader_abs_return"],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    )

    analysis_df["lead_lag_payoff"] = (
        np.sign(analysis_df[leader_return_col])
        * analysis_df[follower_next_return_col]
    )

    result = (
        analysis_df
        .groupby("leader_quintile", observed=True)
        .agg(
            observations=("lead_lag_payoff", "size"),
            mean_lead_lag_payoff=("lead_lag_payoff", "mean"),
            median_lead_lag_payoff=("lead_lag_payoff", "median"),
            win_rate=("lead_lag_payoff", lambda x: (x > 0).mean()),
            min_leader_move=("leader_abs_return", "min"),
            max_leader_move=("leader_abs_return", "max"),
        )
    )

    result["mean_payoff_bps"] = (
        result["mean_lead_lag_payoff"] * 10_000
    )

    result["median_payoff_bps"] = (
        result["median_lead_lag_payoff"] * 10_000
    )

    result["win_rate_pct"] = result["win_rate"] * 100

    return analysis_df, result


rb_leads_hc_data, rb_leads_hc_quintiles = (
    analyze_lead_lag_quintiles(
        lead_lag_train,
        leader_return_col="return_15m_rb",
        follower_next_return_col="next_return_15m_hc",
    )
)

hc_leads_rb_data, hc_leads_rb_quintiles = (
    analyze_lead_lag_quintiles(
        lead_lag_train,
        leader_return_col="return_15m_hc",
        follower_next_return_col="next_return_15m_rb",
    )
)

print("RB leads HC:")
print(
    rb_leads_hc_quintiles[
        [
            "observations",
            "min_leader_move",
            "max_leader_move",
            "mean_payoff_bps",
            "median_payoff_bps",
            "win_rate_pct",
        ]
    ]
)

print("HC leads RB:")
print(
    hc_leads_rb_quintiles[
        [
            "observations",
            "min_leader_move",
            "max_leader_move",
            "mean_payoff_bps",
            "median_payoff_bps",
            "win_rate_pct",
        ]
    ]
)



RB leads HC:
                 observations  min_leader_move  max_leader_move  \
leader_quintile                                                   
Q1                       3772         0.000229         0.000580   
Q2                       3773         0.000580         0.001203   
Q3                       3770         0.001203         0.002024   
Q4                       3772         0.002025         0.003501   
Q5                       3772         0.003503         0.042211   

                 mean_payoff_bps  median_payoff_bps  win_rate_pct  
leader_quintile                                                    
Q1                      0.135351           0.000000     42.338282  
Q2                      0.389917          -0.000000     43.387225  
Q3                      0.666947          -0.000000     43.076923  
Q4                     -1.196408          -2.494395     41.648993  
Q5                     -0.518261          -3.172092     42.152704  
HC leads RB:
                 observation

In [63]:
hc_q5 = hc_leads_rb_data.loc[
    hc_leads_rb_data["leader_quintile"] == "Q5"
].copy()

payoff = hc_q5["lead_lag_payoff"]

observations = len(hc_q5)
positive_count = (payoff > 0).sum()
negative_count = (payoff < 0).sum()
zero_count = (payoff == 0).sum()

mean_positive_bps = payoff.loc[payoff > 0].mean() * 10_000
mean_negative_bps = payoff.loc[payoff < 0].mean() * 10_000

standard_error = payoff.std() / np.sqrt(observations)
t_statistic = payoff.mean() / standard_error

overall_q5_statistics = pd.Series({
    "observations": observations,
    "positive_count": positive_count,
    "negative_count": negative_count,
    "zero_count": zero_count,
    "mean_payoff_bps": payoff.mean() * 10_000,
    "median_payoff_bps": payoff.median() * 10_000,
    "mean_positive_payoff_bps": mean_positive_bps,
    "mean_negative_payoff_bps": mean_negative_bps,
    "win_rate_pct": (payoff > 0).mean() * 100,
    "loss_rate_pct": (payoff < 0).mean() * 100,
    "zero_rate_pct": (payoff == 0).mean() * 100,
    "standard_error_bps": standard_error * 10_000,
    "t_statistic": t_statistic,
})

print("Overall HC Q5 -> RB:")
print(overall_q5_statistics)


hc_q5["year"] = hc_q5["end"].dt.year

yearly_q5 = (
    hc_q5
    .groupby("year")
    .agg(
        observations=("lead_lag_payoff", "size"),
        mean_payoff=("lead_lag_payoff", "mean"),
        median_payoff=("lead_lag_payoff", "median"),
        std_payoff=("lead_lag_payoff", "std"),
        win_rate=(
            "lead_lag_payoff",
            lambda x: (x > 0).mean(),
        ),
        loss_rate=(
            "lead_lag_payoff",
            lambda x: (x < 0).mean(),
        ),
        zero_rate=(
            "lead_lag_payoff",
            lambda x: (x == 0).mean(),
        ),
    )
)

yearly_q5["standard_error"] = (
    yearly_q5["std_payoff"]
    / np.sqrt(yearly_q5["observations"])
)

yearly_q5["t_statistic"] = (
    yearly_q5["mean_payoff"]
    / yearly_q5["standard_error"]
)

yearly_q5["mean_payoff_bps"] = (
    yearly_q5["mean_payoff"] * 10_000
)

yearly_q5["median_payoff_bps"] = (
    yearly_q5["median_payoff"] * 10_000
)

yearly_q5["win_rate_pct"] = yearly_q5["win_rate"] * 100
yearly_q5["loss_rate_pct"] = yearly_q5["loss_rate"] * 100
yearly_q5["zero_rate_pct"] = yearly_q5["zero_rate"] * 100

print("\nYearly HC Q5 -> RB:")
print(
    yearly_q5[
        [
            "observations",
            "mean_payoff_bps",
            "median_payoff_bps",
            "win_rate_pct",
            "loss_rate_pct",
            "zero_rate_pct",
            "t_statistic",
        ]
    ]
)

Overall HC Q5 -> RB:
observations                3712.000000
positive_count              1667.000000
negative_count              1849.000000
zero_count                   196.000000
mean_payoff_bps                1.380569
median_payoff_bps              0.000000
mean_positive_payoff_bps      35.162750
mean_negative_payoff_bps     -28.930034
win_rate_pct                  44.908405
loss_rate_pct                 49.811422
zero_rate_pct                  5.280172
standard_error_bps             0.705727
t_statistic                    1.956236
dtype: float64

Yearly HC Q5 -> RB:
      observations  mean_payoff_bps  median_payoff_bps  win_rate_pct  \
year                                                                   
2015           375         2.014997           0.000000     42.666667   
2016          1421         1.670117          -0.000000     45.601689   
2017          1317         0.390176          -2.864509     44.570995   
2018           599         2.474040           0.000000     45.4

In [64]:
rb_hc = rb_hc.copy()

session_keys = [
    "session_block_rb",
    "session_block_hc",
]

rb_hc["prior_60m_mean_volume_hc"] = (
    rb_hc
    .groupby(session_keys)["volume_hc"]
    .transform(
        lambda x: (
            x.shift(1)
            .rolling(window=4, min_periods=4)
            .mean()
        )
    )
)

rb_hc["volume_ratio_hc"] = (
    rb_hc["volume_hc"]
    / rb_hc["prior_60m_mean_volume_hc"]
)

hc_q5 = hc_leads_rb_data.loc[
    hc_leads_rb_data["leader_quintile"] == "Q5"
].copy()

hc_q5["volume_ratio_hc"] = (
    rb_hc.loc[hc_q5.index, "volume_ratio_hc"]
)

hc_q5 = hc_q5.loc[
    hc_q5["volume_ratio_hc"].notna()
].copy()

hc_q5["volume_quartile"] = pd.qcut(
    hc_q5["volume_ratio_hc"],
    q=4,
    labels=["V1", "V2", "V3", "V4"],
)

volume_statistics = (
    hc_q5
    .groupby("volume_quartile", observed=True)
    .agg(
        observations=("lead_lag_payoff", "size"),
        min_volume_ratio=("volume_ratio_hc", "min"),
        max_volume_ratio=("volume_ratio_hc", "max"),
        mean_payoff=("lead_lag_payoff", "mean"),
        median_payoff=("lead_lag_payoff", "median"),
        std_payoff=("lead_lag_payoff", "std"),
        win_rate=(
            "lead_lag_payoff",
            lambda x: (x > 0).mean(),
        ),
        zero_rate=(
            "lead_lag_payoff",
            lambda x: (x == 0).mean(),
        ),
    )
)

volume_statistics["standard_error"] = (
    volume_statistics["std_payoff"]
    / np.sqrt(volume_statistics["observations"])
)

volume_statistics["t_statistic"] = (
    volume_statistics["mean_payoff"]
    / volume_statistics["standard_error"]
)

volume_statistics["mean_payoff_bps"] = (
    volume_statistics["mean_payoff"] * 10_000
)

volume_statistics["median_payoff_bps"] = (
    volume_statistics["median_payoff"] * 10_000
)

volume_statistics["win_rate_pct"] = (
    volume_statistics["win_rate"] * 100
)

volume_statistics["zero_rate_pct"] = (
    volume_statistics["zero_rate"] * 100
)

print(
    volume_statistics[
        [
            "observations",
            "min_volume_ratio",
            "max_volume_ratio",
            "mean_payoff_bps",
            "median_payoff_bps",
            "win_rate_pct",
            "zero_rate_pct",
            "t_statistic",
        ]
    ]
)

                 observations  min_volume_ratio  max_volume_ratio  \
volume_quartile                                                     
V1                        161          0.093817          0.781014   
V2                        160          0.781108          1.123793   
V3                        160          1.125354          1.718064   
V4                        161          1.723383          7.851852   

                 mean_payoff_bps  median_payoff_bps  win_rate_pct  \
volume_quartile                                                     
V1                      2.015472           0.000000     45.962733   
V2                     -0.877356           0.000000     45.625000   
V3                      9.732286           6.938059     57.500000   
V4                      6.635031           4.103406     52.795031   

                 zero_rate_pct  t_statistic  
volume_quartile                              
V1                    5.590062     0.589017  
V2                    5.625000  

In [65]:
volume_threshold = hc_q5["volume_ratio_hc"].median()

hc_q5["high_volume_signal"] = (
    hc_q5["volume_ratio_hc"] > volume_threshold
)

high_volume_trades = hc_q5.loc[
    hc_q5["high_volume_signal"]
].copy()

payoff = high_volume_trades["lead_lag_payoff"]

observations = len(high_volume_trades)
standard_error = payoff.std() / np.sqrt(observations)

combined_statistics = pd.Series({
    "volume_threshold": volume_threshold,
    "observations": observations,
    "mean_payoff_bps": payoff.mean() * 10_000,
    "median_payoff_bps": payoff.median() * 10_000,
    "mean_positive_payoff_bps": (
        payoff.loc[payoff > 0].mean() * 10_000
    ),
    "mean_negative_payoff_bps": (
        payoff.loc[payoff < 0].mean() * 10_000
    ),
    "win_rate_pct": (payoff > 0).mean() * 100,
    "loss_rate_pct": (payoff < 0).mean() * 100,
    "zero_rate_pct": (payoff == 0).mean() * 100,
    "standard_error_bps": standard_error * 10_000,
    "t_statistic": payoff.mean() / standard_error,
})

print("Combined V3 + V4:")
print(combined_statistics)


high_volume_trades["year"] = (
    high_volume_trades["end"].dt.year
)

yearly_high_volume = (
    high_volume_trades
    .groupby("year")
    .agg(
        observations=("lead_lag_payoff", "size"),
        mean_payoff=("lead_lag_payoff", "mean"),
        median_payoff=("lead_lag_payoff", "median"),
        std_payoff=("lead_lag_payoff", "std"),
        win_rate=(
            "lead_lag_payoff",
            lambda x: (x > 0).mean(),
        ),
        zero_rate=(
            "lead_lag_payoff",
            lambda x: (x == 0).mean(),
        ),
    )
)

yearly_high_volume["standard_error"] = (
    yearly_high_volume["std_payoff"]
    / np.sqrt(yearly_high_volume["observations"])
)

yearly_high_volume["t_statistic"] = (
    yearly_high_volume["mean_payoff"]
    / yearly_high_volume["standard_error"]
)

yearly_high_volume["mean_payoff_bps"] = (
    yearly_high_volume["mean_payoff"] * 10_000
)

yearly_high_volume["median_payoff_bps"] = (
    yearly_high_volume["median_payoff"] * 10_000
)

yearly_high_volume["win_rate_pct"] = (
    yearly_high_volume["win_rate"] * 100
)

yearly_high_volume["zero_rate_pct"] = (
    yearly_high_volume["zero_rate"] * 100
)

print("\nYearly combined V3 + V4:")
print(
    yearly_high_volume[
        [
            "observations",
            "mean_payoff_bps",
            "median_payoff_bps",
            "win_rate_pct",
            "zero_rate_pct",
            "t_statistic",
        ]
    ]
)

Combined V3 + V4:
volume_threshold              1.124573
observations                321.000000
mean_payoff_bps               8.178834
median_payoff_bps             6.770481
mean_positive_payoff_bps     40.892482
mean_negative_payoff_bps    -35.210408
win_rate_pct                 55.140187
loss_rate_pct                40.809969
zero_rate_pct                 4.049844
standard_error_bps            2.916795
t_statistic                   2.804048
dtype: float64

Yearly combined V3 + V4:
      observations  mean_payoff_bps  median_payoff_bps  win_rate_pct  \
year                                                                   
2015            35         9.575589          -0.000000     48.571429   
2016           120        12.497494          15.026231     62.500000   
2017           117         6.171807           0.000000     48.717949   
2018            49         1.397131           7.105637     57.142857   

      zero_rate_pct  t_statistic  
year                              
2015     

In [66]:
hc_move_threshold = (
    hc_leads_rb_data["leader_abs_return"]
    .quantile(0.80)
)

hc_volume_threshold = volume_threshold

lead_lag_validation = rb_hc.loc[
    (rb_hc["end"] >= start_date_validation)
    & (rb_hc["end"] <= end_date_validation)
    & rb_hc["return_15m_hc"].notna()
    & rb_hc["next_return_15m_rb"].notna()
    & rb_hc["volume_ratio_hc"].notna()
].copy()

lead_lag_validation["signal"] = np.where(
    (
        lead_lag_validation["return_15m_hc"].abs()
        > hc_move_threshold
    )
    & (
        lead_lag_validation["volume_ratio_hc"]
        > hc_volume_threshold
    ),
    np.sign(lead_lag_validation["return_15m_hc"]),
    0,
)

validation_trades = lead_lag_validation.loc[
    lead_lag_validation["signal"] != 0
].copy()

validation_trades["lead_lag_payoff"] = (
    validation_trades["signal"]
    * validation_trades["next_return_15m_rb"]
)

validation_payoff = validation_trades["lead_lag_payoff"]

validation_observations = len(validation_trades)

validation_standard_error = (
    validation_payoff.std()
    / np.sqrt(validation_observations)
)

estimated_cost_bps = 4.86

validation_statistics = pd.Series({
    "hc_move_threshold_pct": hc_move_threshold * 100,
    "hc_volume_threshold": hc_volume_threshold,
    "observations": validation_observations,
    "long_trades": (validation_trades["signal"] == 1).sum(),
    "short_trades": (validation_trades["signal"] == -1).sum(),
    "gross_payoff_bps": validation_payoff.mean() * 10_000,
    "estimated_cost_bps": estimated_cost_bps,
    "estimated_net_payoff_bps": (
        validation_payoff.mean() * 10_000
        - estimated_cost_bps
    ),
    "median_payoff_bps": (
        validation_payoff.median() * 10_000
    ),
    "win_rate_pct": (
        (validation_payoff > 0).mean() * 100
    ),
    "loss_rate_pct": (
        (validation_payoff < 0).mean() * 100
    ),
    "zero_rate_pct": (
        (validation_payoff == 0).mean() * 100
    ),
    "standard_error_bps": (
        validation_standard_error * 10_000
    ),
    "t_statistic": (
        validation_payoff.mean()
        / validation_standard_error
    ),
})

print("Fixed train thresholds:")
print("HC movement threshold:", hc_move_threshold)
print("HC volume threshold:", hc_volume_threshold)

print("\n2019 validation:")
print(validation_statistics)

Fixed train thresholds:
HC movement threshold: 0.0035741386429640537
HC volume threshold: 1.1245734660985356

2019 validation:
hc_move_threshold_pct        0.357414
hc_volume_threshold          1.124573
observations                42.000000
long_trades                 20.000000
short_trades                22.000000
gross_payoff_bps             2.796568
estimated_cost_bps           4.860000
estimated_net_payoff_bps    -2.063432
median_payoff_bps            1.418842
win_rate_pct                50.000000
loss_rate_pct               45.238095
zero_rate_pct                4.761905
standard_error_bps           4.481897
t_statistic                  0.623970
dtype: float64


Large HC movements accompanied by above-median relative volume exhibited a positive lead–lag relationship with the subsequent 15-minute RB return during 2015–2018. The filtered signal produced a gross mean payoff of 8.18 bps and a t-statistic of 2.80 on the training sample. However, its 2019 validation payoff declined to 2.80 bps, with an estimated net payoff of −2.06 bps after transaction costs and a t-statistic of 0.62. Therefore, the relationship did not generalize strongly enough to support a tradable strategy in its current form

## Hypothesis
A smooth 60-minute move formed by several 15-minute bars in the same direction is more likely to continue. A move resulting from a single sharp spike is more prone to retracement

In [67]:
discreteness_by_instrument = {}

for instrument, bars_df in bars_by_instrument.items():
    discreteness_df = bars_df.copy()

    discreteness_df["return_15m"] = (
        discreteness_df["close"]
        / discreteness_df["open"]
        - 1
    )

    discreteness_df["next_return_15m"] = (
        discreteness_df
        .groupby("session_block")["return_15m"]
        .shift(-1)
    )

    discreteness_df["past_return_60m"] = (
        discreteness_df
        .groupby("session_block")["return_15m"]
        .transform(
            lambda x: (
                (1 + x)
                .rolling(window=4, min_periods=4)
                .apply(np.prod, raw=True)
                - 1
            )
        )
    )

    discreteness_df["direction_balance_60m"] = (
        discreteness_df
        .groupby("session_block")["return_15m"]
        .transform(
            lambda x: (
                x.rolling(window=4, min_periods=4)
                .apply(
                    lambda values: np.sign(values).mean(),
                    raw=True,
                )
            )
        )
    )

    discreteness_df["smoothness_score"] = (
        np.sign(discreteness_df["past_return_60m"])
        * discreteness_df["direction_balance_60m"]
    )

    discreteness_df["largest_bar_60m"] = (
        discreteness_df
        .groupby("session_block")["return_15m"]
        .transform(
            lambda x: (
                x.abs()
                .rolling(window=4, min_periods=4)
                .max()
            )
        )
    )

    discreteness_df["total_absolute_move_60m"] = (
        discreteness_df
        .groupby("session_block")["return_15m"]
        .transform(
            lambda x: (
                x.abs()
                .rolling(window=4, min_periods=4)
                .sum()
            )
        )
    )

    discreteness_df["move_concentration"] = (
        discreteness_df["largest_bar_60m"]
        / discreteness_df["total_absolute_move_60m"]
    )

    discreteness_by_instrument[instrument] = discreteness_df


rb_discreteness = discreteness_by_instrument["RB"]

print(
    rb_discreteness[
        [
            "end",
            "return_15m",
            "past_return_60m",
            "direction_balance_60m",
            "smoothness_score",
            "move_concentration",
            "next_return_15m",
        ]
    ].dropna().head(10)
)

print("\nRB smoothness distribution:")
print(
    rb_discreteness["smoothness_score"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nRB concentration summary:")
print(
    rb_discreteness["move_concentration"].describe()
)

                   end  return_15m  past_return_60m  direction_balance_60m  \
3  2009-03-27 10:00:00   -0.001670         0.010142                   0.00   
12 2009-03-27 14:30:00   -0.000847        -0.000565                   0.00   
13 2009-03-27 14:45:00    0.005088         0.005941                   0.50   
18 2009-03-30 10:00:00    0.000000        -0.001127                  -0.25   
27 2009-03-30 14:30:00   -0.004230        -0.004511                  -0.25   
28 2009-03-30 14:45:00    0.001416        -0.003101                   0.00   
33 2009-03-31 10:00:00    0.002823         0.003674                   0.50   
42 2009-03-31 14:30:00    0.000000        -0.002812                  -0.75   
43 2009-03-31 14:45:00    0.000000        -0.001970                  -0.50   
48 2009-04-01 10:00:00   -0.000563        -0.003090                  -0.50   

    smoothness_score  move_concentration  next_return_15m  
3               0.00            0.531553         0.002232  
12             -0.00 

In [68]:
discreteness_rows = []

for instrument, discreteness_df in discreteness_by_instrument.items():
    train_df = discreteness_df.loc[
        (discreteness_df["end"] >= start_date_train)
        & (discreteness_df["end"] <= end_date_train)
        & discreteness_df["past_return_60m"].notna()
        & discreteness_df["next_return_15m"].notna()
        & discreteness_df["smoothness_score"].notna()
        & discreteness_df["move_concentration"].notna()
        & discreteness_df["past_return_60m"].ne(0)
    ].copy()

    train_df["abs_past_return_60m"] = (
        train_df["past_return_60m"].abs()
    )

    train_df["move_quintile"] = pd.qcut(
        train_df["abs_past_return_60m"],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    )

    train_q5 = train_df.loc[
        train_df["move_quintile"] == "Q5"
    ].copy()

    train_q5["path_type"] = np.select(
        [
            train_q5["smoothness_score"] >= 0.75,
            train_q5["smoothness_score"] <= 0,
        ],
        [
            "smooth",
            "shock_like",
        ],
        default="mixed",
    )

    train_q5["momentum_payoff"] = (
        np.sign(train_q5["past_return_60m"])
        * train_q5["next_return_15m"]
    )

    instrument_statistics = (
        train_q5
        .groupby("path_type")
        .agg(
            observations=("momentum_payoff", "size"),
            mean_payoff=("momentum_payoff", "mean"),
            median_payoff=("momentum_payoff", "median"),
            mean_concentration=("move_concentration", "mean"),
            win_rate=(
                "momentum_payoff",
                lambda x: (x > 0).mean(),
            ),
        )
        .reset_index()
    )

    instrument_statistics["instrument"] = instrument

    instrument_statistics["mean_payoff_bps"] = (
        instrument_statistics["mean_payoff"] * 10_000
    )

    instrument_statistics["median_payoff_bps"] = (
        instrument_statistics["median_payoff"] * 10_000
    )

    instrument_statistics["win_rate_pct"] = (
        instrument_statistics["win_rate"] * 100
    )

    discreteness_rows.append(instrument_statistics)


discreteness_summary = pd.concat(
    discreteness_rows,
    ignore_index=True,
)

print("Results by instrument:")
print(
    discreteness_summary[
        [
            "instrument",
            "path_type",
            "observations",
            "mean_concentration",
            "mean_payoff_bps",
            "median_payoff_bps",
            "win_rate_pct",
        ]
    ]
)


cross_market_discreteness = (
    discreteness_summary
    .groupby("path_type")
    .agg(
        instrument_count=("instrument", "nunique"),
        total_observations=("observations", "sum"),
        mean_instrument_payoff_bps=(
            "mean_payoff_bps",
            "mean",
        ),
        median_instrument_payoff_bps=(
            "mean_payoff_bps",
            "median",
        ),
        min_instrument_payoff_bps=(
            "mean_payoff_bps",
            "min",
        ),
        max_instrument_payoff_bps=(
            "mean_payoff_bps",
            "max",
        ),
        mean_win_rate_pct=(
            "win_rate_pct",
            "mean",
        ),
    )
)

print("\nCross-market results:")
print(cross_market_discreteness)

Results by instrument:
   instrument   path_type  observations  mean_concentration  mean_payoff_bps  \
0          RB       mixed           958            0.514177         1.746635   
1          RB  shock_like           313            0.580104        -0.672038   
2          RB      smooth           540            0.502937         1.530851   
3          CU       mixed          1459            0.528159         0.141549   
4          CU  shock_like           502            0.597019         1.747314   
5          CU      smooth           877            0.511962         1.208847   
6          AU       mixed          2066            0.529677         0.058152   
7          AU  shock_like           671            0.595230        -0.470092   
8          AU      smooth          1199            0.513872         0.356208   
9          RU       mixed           758            0.548088        -0.007859   
10         RU  shock_like           249            0.612396         0.018485   
11         RU    

In [70]:
concentration_rows = []

for instrument, discreteness_df in discreteness_by_instrument.items():
    train_df = discreteness_df.loc[
        (discreteness_df["end"] >= start_date_train)
        & (discreteness_df["end"] <= end_date_train)
        & discreteness_df["past_return_60m"].notna()
        & discreteness_df["next_return_15m"].notna()
        & discreteness_df["move_concentration"].notna()
        & discreteness_df["past_return_60m"].ne(0)
    ].copy()

    train_df["abs_past_return_60m"] = (
        train_df["past_return_60m"].abs()
    )

    train_df["move_quintile"] = pd.qcut(
        train_df["abs_past_return_60m"],
        q=5,
        labels=["Q1", "Q2", "Q3", "Q4", "Q5"],
    )

    train_q5 = train_df.loc[
        train_df["move_quintile"] == "Q5"
    ].copy()

    train_q5["concentration_quintile"] = pd.qcut(
        train_q5["move_concentration"],
        q=5,
        labels=["C1", "C2", "C3", "C4", "C5"],
    )

    train_q5["momentum_payoff"] = (
        np.sign(train_q5["past_return_60m"])
        * train_q5["next_return_15m"]
    )

    instrument_concentration = (
        train_q5
        .groupby(
            "concentration_quintile",
            observed=True,
        )
        .agg(
            observations=("momentum_payoff", "size"),
            min_concentration=("move_concentration", "min"),
            max_concentration=("move_concentration", "max"),
            mean_payoff=("momentum_payoff", "mean"),
            median_payoff=("momentum_payoff", "median"),
            std_payoff=("momentum_payoff", "std"),
            win_rate=(
                "momentum_payoff",
                lambda x: (x > 0).mean(),
            ),
        )
        .reset_index()
    )

    instrument_concentration["standard_error"] = (
        instrument_concentration["std_payoff"]
        / np.sqrt(instrument_concentration["observations"])
    )

    instrument_concentration["t_statistic"] = (
        instrument_concentration["mean_payoff"]
        / instrument_concentration["standard_error"]
    )

    instrument_concentration["mean_payoff_bps"] = (
        instrument_concentration["mean_payoff"] * 10_000
    )

    instrument_concentration["median_payoff_bps"] = (
        instrument_concentration["median_payoff"] * 10_000
    )

    instrument_concentration["win_rate_pct"] = (
        instrument_concentration["win_rate"] * 100
    )

    instrument_concentration["instrument"] = instrument

    concentration_rows.append(instrument_concentration)


concentration_summary = pd.concat(
    concentration_rows,
    ignore_index=True,
)

print("By instrument:")
print(
    concentration_summary[
        [
            "instrument",
            "concentration_quintile",
            "observations",
            "min_concentration",
            "max_concentration",
            "mean_payoff_bps",
            "median_payoff_bps",
            "win_rate_pct",
            "t_statistic",
        ]
    ]
)


cross_market_concentration = (
    concentration_summary
    .groupby("concentration_quintile", observed=True)
    .agg(
        instrument_count=("instrument", "nunique"),
        total_observations=("observations", "sum"),
        mean_instrument_payoff_bps=(
            "mean_payoff_bps",
            "mean",
        ),
        median_instrument_payoff_bps=(
            "mean_payoff_bps",
            "median",
        ),
        min_instrument_payoff_bps=(
            "mean_payoff_bps",
            "min",
        ),
        max_instrument_payoff_bps=(
            "mean_payoff_bps",
            "max",
        ),
        mean_win_rate_pct=(
            "win_rate_pct",
            "mean",
        ),
    )
)

print("\nCross-market concentration results:")
print(cross_market_concentration)

By instrument:
   instrument concentration_quintile  observations  min_concentration  \
0          RB                     C1           363           0.266692   
1          RB                     C2           362           0.408850   
2          RB                     C3           362           0.468508   
3          RB                     C4           362           0.539736   
4          RB                     C5           362           0.638123   
5          CU                     C1           568           0.263572   
6          CU                     C2           567           0.417138   
7          CU                     C3           568           0.480231   
8          CU                     C4           567           0.557294   
9          CU                     C5           568           0.653679   
10         AU                     C1           788           0.266899   
11         AU                     C2           787           0.416540   
12         AU                     C3